# Business Understanding

### Background
Maritime transport is a major component of global trade, connecting economies through shipping routes, ports and logistics networks. Understanding how trade volumes, shipping capacity, connectivity and port activity interact is important for analysing the performance and evolution of the global maritime system.

### Problem Statement
Maritime trade data is distributed across multiple sources covering different aspects of the shipping system. Analysing these indicators separately limits the understanding of their relationships, emerging patterns and changes during major disruptions. An integrated analytical approach can provide broader insights into global maritime trade and its future trends.

### Objectives
**General Objective:**  
To analyse global maritime trade patterns, relationships and emerging trends using data analytics, statistical analysis and machine learning.

**Specific Objectives:**
1. Examine historical trends in global maritime trade and related indicators.
2. Investigate relationships among trade, shipping capacity, connectivity and port activity.
3. Analyse the effects of major disruptions on maritime trade and shipping activity.
4. Assess maritime resilience through disruption impact, recovery and stability.
5. Identify unusual port activity and group ports by operational patterns.
6. Develop exploratory forecasts of annual seaborne trade by cargo type and geographical region, where data permits.

### Research Questions
1. How have global maritime trade patterns evolved over time?
2. What relationships exist among trade, shipping capacity, connectivity and port activity?
3. How have major disruptions affected maritime trade and shipping activity?
4. How does the maritime system respond and recover from disruptions?
5. What similarities and unusual activity patterns can be identified across ports?
6. What future trends in seaborne trade by cargo type and region can be estimated?

### Scope and Expected Outcomes
The project uses eight datasets covering seaborne trade, container throughput, merchant fleets, connectivity, port calls, port performance, daily port activity and chokepoint transits.

Expected outcomes include insights into maritime trade patterns, relationships, disruption impacts, port activity and exploratory trade forecasts. Forecasts will be evaluated against a naïve baseline. Findings describe associations rather than establish causation, and forecasts are subject to data coverage and limitations.


# Data Understanding

In [127]:
# imports 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from scipy.stats import pearsonr, spearmanr
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from statsmodels.tsa.arima.model import ARIMA
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (r2_score,mean_absolute_error,mean_squared_error,mean_absolute_percentage_error,silhouette_score,davies_bouldin_score,calinski_harabasz_score)
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
from pathlib import Path
import pycountry

PROC = Path("Datasets/processed")
PROC.mkdir(parents=True, exist_ok=True)

In [128]:
# load daily ports
portwatch = pd.read_csv("Datasets/raw/daily_ports_all.csv")
print(portwatch.shape)
portwatch.head()

(5952054, 30)


,date,year,month,day,portid,portname,country,ISO3,portcalls_container,portcalls_dry_bulk,...,import_cargo,import,export_container,export_dry_bulk,export_general_cargo,export_roro,export_tanker,export_cargo,export,ObjectId
0,2021-03-11,2021,3,11,port1325,Tsuruga,Japan,JPN,0,0,...,0,0,0,0,0,0,0,0,0,1
1,2020-02-05,2020,2,5,port1325,Tsuruga,Japan,JPN,0,0,...,160,160,0,0,0,543,0,543,543,2
2,2021-03-12,2021,3,12,port1325,Tsuruga,Japan,JPN,0,1,...,242,242,0,1754,0,0,0,1754,1754,3
3,2021-03-13,2021,3,13,port1325,Tsuruga,Japan,JPN,0,0,...,0,0,0,0,0,0,0,0,0,4
4,2021-03-14,2021,3,14,port1325,Tsuruga,Japan,JPN,0,1,...,320,320,0,2670,0,1087,0,3758,3758,5


In [129]:
portwatch.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5952054 entries, 0 to 5952053
Data columns (total 30 columns):
 #   Column                   Dtype 
---  ------                   ----- 
 0   date                     object
 1   year                     int64 
 2   month                    int64 
 3   day                      int64 
 4   portid                   object
 5   portname                 object
 6   country                  object
 7   ISO3                     object
 8   portcalls_container      int64 
 9   portcalls_dry_bulk       int64 
 10  portcalls_general_cargo  int64 
 11  portcalls_roro           int64 
 12  portcalls_tanker         int64 
 13  portcalls_cargo          int64 
 14  portcalls                int64 
 15  import_container         int64 
 16  import_dry_bulk          int64 
 17  import_general_cargo     int64 
 18  import_roro              int64 
 19  import_tanker            int64 
 20  import_cargo             int64 
 21  import                   int64 

In [130]:
portwatch.describe()

,year,month,day,portcalls_container,portcalls_dry_bulk,portcalls_general_cargo,portcalls_roro,portcalls_tanker,portcalls_cargo,portcalls,...,import_cargo,import,export_container,export_dry_bulk,export_general_cargo,export_roro,export_tanker,export_cargo,export,ObjectId
count,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,...,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06
mean,2.022441e+03,6.437370e+00,1.568963e+01,6.170749e-01,3.757427e-01,5.177514e-01,1.377770e-01,7.559642e-01,1.648346e+00,2.404310e+00,...,1.121031e+04,1.593580e+04,4.275932e+03,6.295131e+03,4.177303e+02,6.763786e+01,4.420149e+03,1.105650e+04,1.547669e+04,2.976028e+06
std,2.264716e+00,3.412146e+00,8.803135e+00,2.478460e+00,1.379177e+00,1.478039e+00,5.896698e-01,2.852803e+00,4.420998e+00,6.530501e+00,...,5.532706e+04,6.991363e+04,3.671674e+04,4.729657e+04,1.743020e+03,5.592610e+02,2.778433e+04,6.194919e+04,7.134544e+04,1.718210e+06
min,2.019000e+03,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
25%,2.020000e+03,3.000000e+00,8.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.488014e+06
50%,2.022000e+03,6.000000e+00,1.600000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.976028e+06
75%,2.024000e+03,9.000000e+00,2.300000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,2.000000e+00,2.000000e+00,...,1.116000e+03,3.014000e+03,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,7.180000e+02,2.207000e+03,4.464041e+06
max,2.026000e+03,1.200000e+01,3.100000e+01,7.600000e+01,8.300000e+01,4.300000e+01,1.900000e+01,1.170000e+02,1.760000e+02,2.180000e+02,...,3.406220e+06,3.706936e+06,2.004273e+06,2.641160e+06,8.454700e+04,3.543000e+04,5.437958e+06,2.657613e+06,5.877828e+06,5.952054e+06


In [131]:
## check portwatch mising values
portwatch.isnull().sum()

date                       0
year                       0
month                      0
day                        0
portid                     0
portname                   0
country                    0
ISO3                       0
portcalls_container        0
portcalls_dry_bulk         0
portcalls_general_cargo    0
portcalls_roro             0
portcalls_tanker           0
portcalls_cargo            0
portcalls                  0
import_container           0
import_dry_bulk            0
import_general_cargo       0
import_roro                0
import_tanker              0
import_cargo               0
import                     0
export_container           0
export_dry_bulk            0
export_general_cargo       0
export_roro                0
export_tanker              0
export_cargo               0
export                     0
ObjectId                   0
dtype: int64

In [132]:
portwatch.duplicated().sum()

np.int64(0)

In [133]:
# load us seaborne
seaborne = pd.read_csv('Datasets/raw/US.SeaborneTrade_20261110_095251.csv')
print(seaborne.shape)
seaborne.head()

(333, 77)


,Economy_Label,CargoType_Label,2000_Metric_tons_in_thousands_Value,2000_Metric_tons_Footnote,2000_Metric_tons_MissingValue,2001_Metric_tons_in_thousands_Value,2001_Metric_tons_Footnote,2001_Metric_tons_MissingValue,2002_Metric_tons_in_thousands_Value,2002_Metric_tons_Footnote,...,2021_Metric_tons_MissingValue,2022_Metric_tons_in_thousands_Value,2022_Metric_tons_Footnote,2022_Metric_tons_MissingValue,2023_Metric_tons_in_thousands_Value,2023_Metric_tons_Footnote,2023_Metric_tons_MissingValue,2024_Metric_tons_in_thousands_Value,2024_Metric_tons_Footnote,2024_Metric_tons_MissingValue
0,World,Crude oil loaded,1812768.0,NaN,NaN,1819232.0,NaN,NaN,1788715.0,NaN,...,NaN,2029559.0,NaN,NaN,2071488.0,NaN,NaN,2042568.0,NaN,NaN
1,World,Other tanker trade loaded,1171172.0,NaN,NaN,1162476.0,NaN,NaN,1196826.0,NaN,...,NaN,2290940.0,NaN,NaN,2288012.0,NaN,NaN,2284711.0,NaN,NaN
2,World,Dry cargo loaded,3361790.0,NaN,NaN,3416510.0,NaN,NaN,3591200.0,NaN,...,NaN,7433784.0,NaN,NaN,7239276.0,NaN,NaN,7726790.0,NaN,NaN
3,World,Total goods loaded,6345730.0,NaN,NaN,6398217.0,NaN,NaN,6576741.0,NaN,...,NaN,11754284.0,NaN,NaN,11598777.0,NaN,NaN,12054069.0,NaN,NaN
4,World,Crude oil discharged,1812768.0,NaN,NaN,1819232.0,NaN,NaN,1788715.0,NaN,...,NaN,2029559.0,NaN,NaN,2071488.0,NaN,NaN,2042568.0,NaN,NaN


In [134]:
seaborne.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 333 entries, 0 to 332
Data columns (total 77 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Economy_Label                        333 non-null    object 
 1   CargoType_Label                      333 non-null    object 
 2   2000_Metric_tons_in_thousands_Value  323 non-null    float64
 3   2000_Metric_tons_Footnote            0 non-null      float64
 4   2000_Metric_tons_MissingValue        9 non-null      object 
 5   2001_Metric_tons_in_thousands_Value  322 non-null    float64
 6   2001_Metric_tons_Footnote            0 non-null      float64
 7   2001_Metric_tons_MissingValue        9 non-null      object 
 8   2002_Metric_tons_in_thousands_Value  324 non-null    float64
 9   2002_Metric_tons_Footnote            0 non-null      float64
 10  2002_Metric_tons_MissingValue        9 non-null      object 
 11  2003_Metric_tons_in_thousands_Va

In [135]:
seaborne.describe()

,2000_Metric_tons_in_thousands_Value,2000_Metric_tons_Footnote,2001_Metric_tons_in_thousands_Value,2001_Metric_tons_Footnote,2002_Metric_tons_in_thousands_Value,2002_Metric_tons_Footnote,2003_Metric_tons_in_thousands_Value,2003_Metric_tons_Footnote,2004_Metric_tons_in_thousands_Value,2004_Metric_tons_Footnote,...,2020_Metric_tons_in_thousands_Value,2020_Metric_tons_Footnote,2021_Metric_tons_in_thousands_Value,2021_Metric_tons_Footnote,2022_Metric_tons_in_thousands_Value,2022_Metric_tons_Footnote,2023_Metric_tons_in_thousands_Value,2023_Metric_tons_Footnote,2024_Metric_tons_in_thousands_Value,2024_Metric_tons_Footnote
count,3.230000e+02,0.0,3.220000e+02,0.0,3.240000e+02,0.0,3.220000e+02,0.0,3.230000e+02,0.0,...,3.230000e+02,0.0,3.240000e+02,0.0,3.240000e+02,0.0,3.240000e+02,0.0,3.240000e+02,0.0
mean,9.871586e+05,NaN,9.974137e+05,NaN,1.019340e+06,NaN,1.080934e+06,NaN,1.166356e+06,NaN,...,1.830849e+06,NaN,1.900331e+06,NaN,1.906340e+06,NaN,1.883756e+06,NaN,1.966969e+06,NaN
std,1.533092e+06,NaN,1.545379e+06,NaN,1.585122e+06,NaN,1.669515e+06,NaN,1.801689e+06,NaN,...,2.829516e+06,NaN,2.939173e+06,NaN,2.938670e+06,NaN,2.901268e+06,NaN,3.051610e+06,NaN
min,1.000000e+00,NaN,0.000000e+00,NaN,1.000000e+00,NaN,0.000000e+00,NaN,1.000000e+00,NaN,...,0.000000e+00,NaN,0.000000e+00,NaN,0.000000e+00,NaN,0.000000e+00,NaN,0.000000e+00,NaN
25%,6.204950e+04,NaN,6.161550e+04,NaN,5.827925e+04,NaN,6.325275e+04,NaN,6.363000e+04,NaN,...,1.232500e+05,NaN,1.228542e+05,NaN,1.261062e+05,NaN,1.285468e+05,NaN,1.338748e+05,NaN
50%,4.190540e+05,NaN,4.285210e+05,NaN,4.487830e+05,NaN,4.721545e+05,NaN,5.050890e+05,NaN,...,7.404920e+05,NaN,7.616535e+05,NaN,7.510270e+05,NaN,7.722395e+05,NaN,7.890845e+05,NaN
75%,1.215596e+06,NaN,1.215856e+06,NaN,1.232880e+06,NaN,1.314100e+06,NaN,1.405542e+06,NaN,...,2.205966e+06,NaN,2.326580e+06,NaN,2.314270e+06,NaN,2.308013e+06,NaN,2.404866e+06,NaN
max,1.269146e+07,NaN,1.279643e+07,NaN,1.315348e+07,NaN,1.386194e+07,NaN,1.498393e+07,NaN,...,2.263529e+07,NaN,2.354976e+07,NaN,2.350857e+07,NaN,2.319755e+07,NaN,2.410814e+07,NaN


In [136]:
# missing values in seaborne
seaborne.isnull().sum()

Economy_Label                            0
CargoType_Label                          0
2000_Metric_tons_in_thousands_Value     10
2000_Metric_tons_Footnote              333
2000_Metric_tons_MissingValue          324
                                      ... 
2023_Metric_tons_Footnote              333
2023_Metric_tons_MissingValue          324
2024_Metric_tons_in_thousands_Value      9
2024_Metric_tons_Footnote              333
2024_Metric_tons_MissingValue          324
Length: 77, dtype: int64

In [137]:
seaborne.duplicated().sum()

np.int64(0)

In [138]:
# load merchant fleet
fleet = pd.read_csv('Datasets/raw/US_MerchantFleet.csv')
print(fleet.shape)
fleet.head()

(63896, 23)


,Year,Economy,Economy Label,ShipType,ShipType Label,Dead weight tons in thousands,Dead weight tons in thousands Footnote,Dead weight tons in thousands Missing value,Percentage of total world,Percentage of total world Footnote,...,Percentage of total fleet Missing value,Number of ships,Number of ships Footnote,Number of ships Missing value,Gross Tonnage in thousands,Gross Tonnage in thousands Footnote,Gross Tonnage in thousands Missing value,Average age of vessels (years),Average age of vessels (years) Footnote,Average age of vessels (years) Missing value
0,1980,0000,World,0,Total fleet,672192.465,NaN,NaN,100.0,NaN,...,NaN,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected
1,1980,0000,World,1,Oil tankers,337945.534,NaN,NaN,100.0,NaN,...,NaN,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected
2,1980,0000,World,2,Bulk carriers,181880.282,NaN,NaN,100.0,NaN,...,NaN,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected
3,1980,0000,World,3,General cargo,112840.897,NaN,NaN,100.0,NaN,...,NaN,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected
4,1980,0000,World,4,Container ships,10290.114,NaN,NaN,100.0,NaN,...,NaN,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected


In [139]:
fleet.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 63896 entries, 0 to 63895
Data columns (total 23 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   Year                                          63896 non-null  int64  
 1   Economy                                       63896 non-null  object 
 2   Economy Label                                 63896 non-null  object 
 3   ShipType                                      63896 non-null  int64  
 4   ShipType Label                                63896 non-null  object 
 5   Dead weight tons in thousands                 58814 non-null  float64
 6   Dead weight tons in thousands Footnote        24 non-null     object 
 7   Dead weight tons in thousands Missing value   5082 non-null   object 
 8   Percentage of total world                     58814 non-null  float64
 9   Percentage of total world Footnote            24 non-null    

In [140]:
fleet.describe()

,Year,ShipType,Dead weight tons in thousands,Percentage of total world,Percentage of total fleet,Number of ships,Gross Tonnage in thousands,Average age of vessels (years)
count,63896.000000,63896.000000,5.881400e+04,58814.000000,58814.000000,24600.000000,2.460000e+04,20023.000000
mean,2004.610852,2.457759,2.174818e+04,5.397872,37.797279,1506.377154,2.017613e+04,24.460755
std,13.783127,1.748450,8.731343e+04,12.848273,37.409220,5927.114144,7.814213e+04,11.648492
min,1980.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,0.000000
25%,1993.000000,1.000000,1.753525e+01,0.008471,5.930415,2.000000,2.652000e+00,16.080000
50%,2006.000000,2.000000,4.118255e+02,0.188330,22.924635,35.000000,1.707100e+02,23.290000
75%,2017.000000,4.000000,6.420298e+03,3.591282,66.899630,543.250000,4.086645e+03,30.906500
max,2026.000000,5.000000,2.528788e+06,100.000000,100.000000,115649.000000,1.737514e+06,109.920000


In [141]:
fleet.isnull().sum()

Year                                                0
Economy                                             0
Economy Label                                       0
ShipType                                            0
ShipType Label                                      0
Dead weight tons in thousands                    5082
Dead weight tons in thousands Footnote          63872
Dead weight tons in thousands Missing value     58814
Percentage of total world                        5082
Percentage of total world Footnote              63872
Percentage of total world Missing value         58814
Percentage of total fleet                        5082
Percentage of total fleet Footnote              63872
Percentage of total fleet Missing value         58814
Number of ships                                 39296
Number of ships Footnote                        63884
Number of ships Missing value                   24600
Gross Tonnage in thousands                      39296
Gross Tonnage in thousands F

In [142]:
fleet.duplicated().sum()

np.int64(0)

In [143]:
# load lsci
lsci = pd.read_csv('Datasets/raw/US.LSCI_20261121_091634.csv')
print(lsci.shape)
lsci.head()

(15013, 5)


,Economy_Label,Quarter_Label,Index_average_Q1_2023__100_Value,Index_average_Q1_2023__100_Footnote,Index_average_Q1_2023__100_MissingValue
0,Albania,Q4 2006,5.01,NaN,NaN
1,Albania,Q1 2007,5.48,NaN,NaN
2,Albania,Q2 2007,5.48,NaN,NaN
3,Albania,Q3 2007,5.04,NaN,NaN
4,Albania,Q4 2007,5.19,NaN,NaN


In [144]:
lsci.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15013 entries, 0 to 15012
Data columns (total 5 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Economy_Label                            15013 non-null  object 
 1   Quarter_Label                            15013 non-null  object 
 2   Index_average_Q1_2023__100_Value         14724 non-null  float64
 3   Index_average_Q1_2023__100_Footnote      0 non-null      float64
 4   Index_average_Q1_2023__100_MissingValue  289 non-null    object 
dtypes: float64(2), object(3)
memory usage: 586.6+ KB


In [145]:
lsci.describe()

,Index_average_Q1_2023__100_Value,Index_average_Q1_2023__100_Footnote
count,14724.000000,0.0
mean,96.932643,NaN
std,133.808579,NaN
min,2.500000,NaN
25%,22.792500,NaN
50%,47.060000,NaN
75%,106.880000,NaN
max,1341.970000,NaN


In [146]:
lsci.isnull().sum()

Economy_Label                                  0
Quarter_Label                                  0
Index_average_Q1_2023__100_Value             289
Index_average_Q1_2023__100_Footnote        15013
Index_average_Q1_2023__100_MissingValue    14724
dtype: int64

In [147]:
lsci.duplicated().sum()

np.int64(0)

In [148]:
# load container throughput
teu = pd.read_csv('Datasets/raw/US_ContPortThroughput.csv')
print(teu.shape)
teu.head()

(3030, 6)


,Year,Economy,Economy Label,TEU (Twenty foot Equivalent Unit),TEU (Twenty foot Equivalent Unit) Footnote,TEU (Twenty foot Equivalent Unit) Missing value
0,2010,0,World,545520944.0,NaN,NaN
1,2010,5100,Africa,23416816.0,NaN,NaN
2,2010,5110,Northern Africa,12024778.0,NaN,NaN
3,2010,5120,Sub-Saharan Africa,11392038.0,NaN,NaN
4,2010,5121,Eastern Africa,2297961.0,NaN,NaN


In [149]:
teu.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3030 entries, 0 to 3029
Data columns (total 6 columns):
 #   Column                                           Non-Null Count  Dtype  
---  ------                                           --------------  -----  
 0   Year                                             3030 non-null   int64  
 1   Economy                                          3030 non-null   int64  
 2   Economy Label                                    3030 non-null   object 
 3   TEU (Twenty foot Equivalent Unit)                2900 non-null   float64
 4   TEU (Twenty foot Equivalent Unit) Footnote       19 non-null     object 
 5   TEU (Twenty foot Equivalent Unit) Missing value  130 non-null    object 
dtypes: float64(1), int64(2), object(3)
memory usage: 142.2+ KB


In [150]:
teu.describe()

,Year,Economy,TEU (Twenty foot Equivalent Unit)
count,3030.000000,3030.000000,2.900000e+03
mean,2017.000000,1431.004950,4.752831e+07
std,4.321207,1819.965223,1.193780e+08
min,2010.000000,0.000000,9.500000e+03
25%,2013.000000,308.000000,4.678590e+05
50%,2017.000000,612.000000,3.420179e+06
75%,2021.000000,1530.000000,2.146032e+07
max,2024.000000,5706.000000,9.203008e+08


In [151]:
teu.isnull().sum()

Year                                                  0
Economy                                               0
Economy Label                                         0
TEU (Twenty foot Equivalent Unit)                   130
TEU (Twenty foot Equivalent Unit) Footnote         3011
TEU (Twenty foot Equivalent Unit) Missing value    2900
dtype: int64

In [152]:
teu.duplicated().sum()

np.int64(0)

In [153]:
# load chokepoints 
choke = pd.read_csv('Datasets/raw/Daily_Chokepoints_Data.csv')
print(choke.shape)
choke.head()

(80528, 21)


,date,year,month,day,portid,portname,n_container,n_dry_bulk,n_general_cargo,n_roro,...,n_cargo,n_total,capacity_container,capacity_dry_bulk,capacity_general_cargo,capacity_roro,capacity_tanker,capacity_cargo,capacity,ObjectId
0,2019/01/01,2019,1,1,chokepoint1,Suez Canal,15,15,9,4,...,43,61,1018466,618654,112660,21900,1279775,1771682,3051458,1
1,2021/03/11,2021,3,11,chokepoint1,Suez Canal,16,12,3,2,...,33,56,964354,351621,20733,10326,1021116,1347035,2368151,2
2,2019/01/02,2019,1,2,chokepoint1,Suez Canal,24,4,6,6,...,40,50,1534924,227315,8536,37934,518306,1808711,2327017,3
3,2019/01/03,2019,1,3,chokepoint1,Suez Canal,13,14,9,2,...,38,62,555097,977689,39398,10829,868092,1583014,2451107,4
4,2021/03/12,2021,3,12,chokepoint1,Suez Canal,22,15,3,2,...,42,54,1372498,752113,2341,11084,670038,2138038,2808077,5


In [154]:
choke.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80528 entries, 0 to 80527
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   date                    80528 non-null  object
 1   year                    80528 non-null  int64 
 2   month                   80528 non-null  int64 
 3   day                     80528 non-null  int64 
 4   portid                  80528 non-null  object
 5   portname                80528 non-null  object
 6   n_container             80528 non-null  int64 
 7   n_dry_bulk              80528 non-null  int64 
 8   n_general_cargo         80528 non-null  int64 
 9   n_roro                  80528 non-null  int64 
 10  n_tanker                80528 non-null  int64 
 11  n_cargo                 80528 non-null  int64 
 12  n_total                 80528 non-null  int64 
 13  capacity_container      80528 non-null  int64 
 14  capacity_dry_bulk       80528 non-null  int64 
 15  ca

In [155]:
choke.describe()

,year,month,day,n_container,n_dry_bulk,n_general_cargo,n_roro,n_tanker,n_cargo,n_total,capacity_container,capacity_dry_bulk,capacity_general_cargo,capacity_roro,capacity_tanker,capacity_cargo,capacity,ObjectId
count,80528.000000,80528.000000,80528.000000,80528.000000,80528.000000,80528.000000,80528.000000,80528.000000,80528.000000,80528.000000,8.052800e+04,8.052800e+04,80528.000000,80528.000000,8.052800e+04,8.052800e+04,8.052800e+04,80528.000000
mean,2022.443672,6.440542,15.688804,14.387666,21.664167,11.469278,2.315766,20.811072,49.836877,70.647949,3.667380e+05,1.097857e+06,38229.182036,5440.079053,5.997501e+05,1.508265e+06,2.108016e+06,40264.500000
std,2.265885,3.413100,8.800194,20.696034,23.221566,15.890087,3.221351,22.203804,51.798241,71.148095,5.214239e+05,1.282904e+06,48834.982214,8670.923652,8.209306e+05,1.581379e+06,2.105823e+06,23246.575576
min,2019.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,1.000000
25%,2020.000000,3.000000,8.000000,1.000000,4.000000,1.000000,0.000000,4.000000,12.000000,16.000000,1.158750e+04,1.355515e+05,3470.000000,0.000000,3.459850e+04,2.281238e+05,2.912115e+05,20132.750000
50%,2022.000000,6.000000,16.000000,6.000000,13.000000,4.000000,1.000000,12.000000,32.000000,46.000000,1.007985e+05,5.545225e+05,17303.500000,0.000000,2.563505e+05,8.743895e+05,1.406020e+06,40264.500000
75%,2024.000000,9.000000,23.000000,17.000000,32.000000,17.000000,3.000000,30.000000,66.000000,95.000000,5.417912e+05,1.732328e+06,55880.250000,7942.500000,8.677295e+05,2.327564e+06,3.258355e+06,60396.250000
max,2026.000000,12.000000,31.000000,172.000000,245.000000,149.000000,23.000000,148.000000,352.000000,472.000000,4.016066e+06,1.317163e+07,388902.000000,83176.000000,6.596353e+06,1.434564e+07,1.632013e+07,80528.000000


In [156]:
choke.isnull().sum()

date                      0
year                      0
month                     0
day                       0
portid                    0
portname                  0
n_container               0
n_dry_bulk                0
n_general_cargo           0
n_roro                    0
n_tanker                  0
n_cargo                   0
n_total                   0
capacity_container        0
capacity_dry_bulk         0
capacity_general_cargo    0
capacity_roro             0
capacity_tanker           0
capacity_cargo            0
capacity                  0
ObjectId                  0
dtype: int64

In [157]:
choke.duplicated().sum()

np.int64(0)

In [158]:
# load port calls
calls = pd.read_csv('Datasets/raw/US_PortCallsArrivals_S.csv')
print(calls.shape)
calls.head()

(9300, 9)


,Period,Period Label,Economy,Economy Label,CommercialMarket,CommercialMarket Label,Number of port calls,Number of port calls Footnote,Number of port calls Missing value
0,2018S01,S1 2018,1,Individual economies,0,All ships,NaN,NaN,Not applicable
1,2018S01,S1 2018,1,Individual economies,2,Liquid bulk carriers,NaN,NaN,Not applicable
2,2018S01,S1 2018,1,Individual economies,7,Liquefied petroleum gas carriers,NaN,NaN,Not applicable
3,2018S01,S1 2018,1,Individual economies,8,Liquefied natural gas carriers,NaN,NaN,Not applicable
4,2018S01,S1 2018,1,Individual economies,5,Dry bulk carriers,NaN,NaN,Not applicable


In [159]:
calls.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9300 entries, 0 to 9299
Data columns (total 9 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Period                              9300 non-null   object 
 1   Period Label                        9300 non-null   object 
 2   Economy                             9300 non-null   int64  
 3   Economy Label                       9300 non-null   object 
 4   CommercialMarket                    9300 non-null   int64  
 5   CommercialMarket Label              9300 non-null   object 
 6   Number of port calls                8810 non-null   float64
 7   Number of port calls Footnote       0 non-null      float64
 8   Number of port calls Missing value  490 non-null    object 
dtypes: float64(2), int64(2), object(5)
memory usage: 654.0+ KB


In [160]:
calls.describe()

,Economy,CommercialMarket,Number of port calls,Number of port calls Footnote
count,9300.000000,9300.000000,8.810000e+03,0.0
mean,2821.193011,3.962903,6.490234e+04,NaN
std,2112.189066,2.568352,1.991938e+05,NaN
min,0.000000,0.000000,1.100000e+01,NaN
25%,842.000000,2.000000,1.380500e+03,NaN
50%,1640.000000,4.000000,7.185000e+03,NaN
75%,5221.000000,6.000000,4.047450e+04,NaN
max,5900.000000,8.000000,2.461711e+06,NaN


In [161]:
choke.isnull().sum()

date                      0
year                      0
month                     0
day                       0
portid                    0
portname                  0
n_container               0
n_dry_bulk                0
n_general_cargo           0
n_roro                    0
n_tanker                  0
n_cargo                   0
n_total                   0
capacity_container        0
capacity_dry_bulk         0
capacity_general_cargo    0
capacity_roro             0
capacity_tanker           0
capacity_cargo            0
capacity                  0
ObjectId                  0
dtype: int64

In [162]:
choke.duplicated().sum()

np.int64(0)

In [163]:
# load cppi 
cppi = pd.read_excel('Datasets/raw/CPPI 2020-2025.xlsx')
print(cppi.shape)
cppi.head()

(426, 20)


,Port,UN/LOCODE,Territory,World Bank Region,Region average CPPI,World Bank Income Group,Income Group average CPPI,Maritime services region,Region's average cppi,CPPI 2020,CPPI 2021,CPPI 2022,CPPI 2023,CPPI 2024,CPPI 2025,Rank 2025,Berth hours in % of port hours,Statistical Index,Administrative Index,Number of calls in sample
0,Aarhus,DKAAR,Denmark,ECA,-8.086595,HI,1.362073,North Europe,-7.718713,73.0,38.0,22.0,60.0,99.0,86.634702,33.0,0.856896,47.61,125.659404,292.0
1,Abidjan,CIABJ,Côte d’Ivoire,SSA,-71.972350,LMI,-8.954771,West Africa,-66.152693,19.0,-153.0,-144.0,-38.0,-51.0,-127.472673,381.0,0.595764,-70.25,-184.695347,407.0
2,Acajutla,SVAQJ,El Salvador,LAC,-14.986492,UMI,1.615451,Caribbean & Central America,-19.239499,0.0,-7.0,-30.0,-89.0,-98.0,-207.620361,393.0,0.479969,-143.38,-271.860723,71.0
3,Adelaide,AUADL,Australia,EAP,30.729639,HI,1.362073,Australasia & Oceania,-11.011090,-20.0,-2.0,-24.0,-39.0,-8.0,-18.214819,297.0,0.833064,-11.43,-24.999638,213.0
4,Aden,YEADE,Yemen,MENAAP,18.958816,LI,-23.905865,Red Sea,63.164190,NaN,-21.0,-13.0,4.0,-24.0,NaN,NaN,NaN,NaN,NaN,NaN


In [164]:
cppi.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 426 entries, 0 to 425
Data columns (total 20 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Port                            424 non-null    object 
 1   UN/LOCODE                       419 non-null    object 
 2   Territory                       419 non-null    object 
 3   World Bank Region               419 non-null    object 
 4   Region average CPPI             419 non-null    float64
 5   World Bank Income Group         419 non-null    object 
 6   Income Group average CPPI       419 non-null    float64
 7   Maritime services region        419 non-null    object 
 8   Region's average cppi           419 non-null    float64
 9   CPPI 2020                       326 non-null    float64
 10  CPPI 2021                       352 non-null    float64
 11  CPPI 2022                       340 non-null    float64
 12  CPPI 2023                       391 

In [165]:
cppi.describe()

,Region average CPPI,Income Group average CPPI,Region's average cppi,CPPI 2020,CPPI 2021,CPPI 2022,CPPI 2023,CPPI 2024,CPPI 2025,Rank 2025,Berth hours in % of port hours,Statistical Index,Administrative Index,Number of calls in sample
count,419.000000,419.000000,419.000000,326.000000,352.000000,340.000000,391.000000,403.000000,400.000000,400.000000,400.000000,400.000000,400.000000,400.000000
mean,-1.341200,-1.359779,-0.973744,37.710675,5.373509,-2.009647,6.866905,0.012407,-1.288518,200.500000,0.756255,-1.871100,-0.705937,449.247500
std,29.907600,6.021963,36.738435,48.038120,80.082824,78.582603,68.823008,71.335361,64.520553,115.614301,0.136932,38.801558,90.544433,782.571439
min,-71.972350,-25.469787,-98.080938,-108.000000,-665.000000,-675.000000,-519.000000,-721.000000,-302.583415,1.000000,0.369475,-176.720000,-428.446829,24.000000
25%,-14.986492,1.362073,-19.239499,6.000000,-7.000000,-16.000000,-12.500000,-16.500000,-19.752567,100.750000,0.673786,-12.080000,-25.612862,81.750000
50%,-8.086595,1.362073,-7.718713,24.000000,12.000000,7.860000,6.000000,4.000000,1.371539,200.500000,0.785014,1.000000,2.509513,194.000000
75%,30.729639,1.615451,17.910438,68.000000,39.000000,34.000000,39.000000,27.000000,24.013080,300.250000,0.860420,13.542500,33.262400,457.000000
max,34.424249,1.615451,103.168045,170.000000,152.000000,139.000000,152.000000,146.000000,144.638476,400.000000,0.983194,83.550000,209.356951,7385.000000


In [166]:
cppi.isnull().sum()

Port                                2
UN/LOCODE                           7
Territory                           7
World Bank Region                   7
Region average CPPI                 7
World Bank Income Group             7
Income Group average CPPI           7
Maritime services region            7
Region's average cppi               7
CPPI 2020                         100
CPPI 2021                          74
CPPI 2022                          86
CPPI 2023                          35
CPPI 2024                          23
CPPI 2025                          26
Rank 2025                          26
Berth hours in % of port hours     26
Statistical Index                  26
Administrative Index               26
Number of calls in sample          26
dtype: int64

In [167]:
cppi.duplicated().sum()

np.int64(1)

# Data Preparation.

####  IMF Portwatch

In [168]:
portwatch.isnull().sum()

date                       0
year                       0
month                      0
day                        0
portid                     0
portname                   0
country                    0
ISO3                       0
portcalls_container        0
portcalls_dry_bulk         0
portcalls_general_cargo    0
portcalls_roro             0
portcalls_tanker           0
portcalls_cargo            0
portcalls                  0
import_container           0
import_dry_bulk            0
import_general_cargo       0
import_roro                0
import_tanker              0
import_cargo               0
import                     0
export_container           0
export_dry_bulk            0
export_general_cargo       0
export_roro                0
export_tanker              0
export_cargo               0
export                     0
ObjectId                   0
dtype: int64

In [169]:
# Inspect
print("Shape:", portwatch.shape)
print("Ports:", portwatch.portid.nunique(), "| countries:", portwatch.country.nunique(), "| ISO3:", portwatch.ISO3.nunique())
print("Missing values:", int(portwatch.isnull().sum().sum()))
print("Duplicate port-day records:", portwatch.duplicated(["portid", "date"]).sum(), "| fully duplicated rows:", portwatch.duplicated().sum())

Shape: (5952054, 30)
Ports: 2071 | countries: 180 | ISO3: 180
Missing values: 0
Duplicate port-day records: 0 | fully duplicated rows: 0


In [170]:
# Dates
portwatch["date"] = pd.to_datetime(portwatch["date"])
parts_match = (portwatch.date.dt.year.eq(portwatch.year) & portwatch.date.dt.month.eq(portwatch.month) & portwatch.date.dt.day.eq(portwatch.day)).mean()
print("year, month, day agree with date:", round(parts_match, 4))
print("period:", portwatch.date.min().date(), "-", portwatch.date.max().date())

year, month, day agree with date: 1.0
period: 2019-01-01 - 2026-11-13


In [171]:
# Panel completeness: every port should have every day
days = portwatch.date.nunique()
per_port = portwatch.groupby("portid").date.agg(["min", "max", "nunique"])
print("days in period:", days, "| ports with every day:", int((per_port["nunique"] == days).sum()), "of", len(per_port))
print("expected rows if complete:", len(per_port) * days, "| actual:", len(portwatch))
print(per_port["nunique"].describe().round(0).to_dict())


days in period: 2874 | ports with every day: 2071 of 2071
expected rows if complete: 5952054 | actual: 5952054
{'count': 2071.0, 'mean': 2874.0, 'std': 0.0, 'min': 2874.0, '25%': 2874.0, '50%': 2874.0, '75%': 2874.0, 'max': 2874.0}


In [172]:
# One name and one country per port
meta = portwatch.groupby("portid")[["portname", "country", "ISO3"]].nunique()
print("ports with more than one name / country / ISO3:", (meta > 1).sum().to_dict())
print("ports without ISO3:", int(portwatch.drop_duplicates("portid").ISO3.isna().sum()))

ports with more than one name / country / ISO3: {'portname': 0, 'country': 0, 'ISO3': 0}
ports without ISO3: 0


In [173]:
# How the totals are built
types5 = ["container", "dry_bulk", "general_cargo", "roro", "tanker"]
types4 = ["container", "dry_bulk", "general_cargo", "roro"]
for prefix in ["portcalls", "import", "export"]:
    five = portwatch[[f"{prefix}_{t}" for t in types5]].sum(axis=1)
    four = portwatch[[f"{prefix}_{t}" for t in types4]].sum(axis=1)
    print(f"{prefix:10} total = 5 types: {(portwatch[prefix] == five).mean():.4f} | cargo = 4 types (no tankers): {(portwatch[prefix + '_cargo'] == four).mean():.4f} | cargo = total: {(portwatch[prefix + '_cargo'] == portwatch[prefix]).mean():.4f}")

portcalls  total = 5 types: 1.0000 | cargo = 4 types (no tankers): 1.0000 | cargo = total: 0.7290
import     total = 5 types: 0.8877 | cargo = 4 types (no tankers): 0.9303 | cargo = total: 0.8133
export     total = 5 types: 0.9044 | cargo = 4 types (no tankers): 0.9394 | cargo = total: 0.8356


In [174]:
# Values
num = portwatch.select_dtypes("number").drop(columns=["year", "month", "day", "ObjectId"])
print("negative values:", int((num < 0).sum().sum()), "| non-finite:", int((~np.isfinite(num)).sum().sum()))
print("port-days with no calls:", round(float(portwatch.portcalls.eq(0).mean()), 3))
print("portcalls per port-day: max", portwatch.portcalls.max(), "| 99.9th percentile", portwatch.portcalls.quantile(.999))
print(portwatch.nlargest(3, "portcalls")[["date", "portname", "country", "portcalls"]].to_string(index=False))

negative values: 0 | non-finite: 0
port-days with no calls: 0.482
portcalls per port-day: max 218 | 99.9th percentile 95.0
      date portname country  portcalls
2024-03-17 Shanghai   China        218
2024-04-30 Shanghai   China        212
2019-05-18 Shanghai   China        196


In [175]:
# Partial last year and ports with no activity
latest_year = portwatch.date.dt.year.max()
portwatch["partial_year"] = portwatch.date.dt.year.eq(latest_year)
print("partial year:", latest_year, "| last date:", portwatch.date.max().date())

tot = portwatch.groupby("portid").portcalls.sum()
print("ports with no calls in the whole period:", int((tot == 0).sum()))

partial year: 2026 | last date: 2026-11-13
ports with no calls in the whole period: 25


In [176]:
# Tidy and save
portwatch_clean = portwatch.drop(columns=["year", "month", "day", "ObjectId"]).sort_values(["portid", "date"]).reset_index(drop=True)
assert not portwatch_clean.duplicated(["portid", "date"]).any()
print("Cleaned shape:", portwatch_clean.shape)

portwatch_clean.to_parquet(PROC / "portwatch_cleaned.parquet", index=False)
port_list = portwatch_clean.drop_duplicates("portid")[["portid", "portname", "country", "ISO3"]]
port_list.to_csv(PROC / "portwatch_ports.csv", index=False)
print("saved:", portwatch_clean.shape, "| ports:", len(port_list))

Cleaned shape: (5952054, 27)
saved: (5952054, 27) | ports: 2071


First look at the full daily file: size, ports, missing values, duplicate port-days.
`date` converted; year, month and day agree with it.
Each port is checked for a complete run of days.
Each port ID maps to one name, one country and one ISO3.
`portcalls` is the sum of five vessel types; the cargo columns exclude tankers; both are kept.
No negative or non-finite values; the busiest port-days are listed to check they are plausible.
2026 is flagged as a partial year (data ends 13 November); ports with no calls are counted.
Redundant columns dropped, data sorted, duplicates checked, saved as Parquet with a port list.

#### US.Seaborne trade UNCTAD

In [177]:
# Seaborne
seaborne.columns

Index(['Economy_Label', 'CargoType_Label',
       '2000_Metric_tons_in_thousands_Value', '2000_Metric_tons_Footnote',
       '2000_Metric_tons_MissingValue', '2001_Metric_tons_in_thousands_Value',
       '2001_Metric_tons_Footnote', '2001_Metric_tons_MissingValue',
       '2002_Metric_tons_in_thousands_Value', '2002_Metric_tons_Footnote',
       '2002_Metric_tons_MissingValue', '2003_Metric_tons_in_thousands_Value',
       '2003_Metric_tons_Footnote', '2003_Metric_tons_MissingValue',
       '2004_Metric_tons_in_thousands_Value', '2004_Metric_tons_Footnote',
       '2004_Metric_tons_MissingValue', '2005_Metric_tons_in_thousands_Value',
       '2005_Metric_tons_Footnote', '2005_Metric_tons_MissingValue',
       '2006_Metric_tons_in_thousands_Value', '2006_Metric_tons_Footnote',
       '2006_Metric_tons_MissingValue', '2007_Metric_tons_in_thousands_Value',
       '2007_Metric_tons_Footnote', '2007_Metric_tons_MissingValue',
       '2008_Metric_tons_in_thousands_Value', '2008_Metric_tons_F

In [178]:
# Create a copy of the original dataset
seaborne_clean = seaborne.copy()

# Identify the annual trade-value columns
value_cols = [col for col in seaborne_clean.columns
    if col.endswith("_Metric_tons_in_thousands_Value")
]

# Reshape from wide format to long format
seaborne_clean = seaborne_clean.melt(id_vars=["Economy_Label", "CargoType_Label"],value_vars=value_cols,var_name="Year",value_name="Trade_thousand_tonnes")

# Extract year from column names
seaborne_clean["Year"] = (seaborne_clean["Year"].str.extract(r"(\d{4})").astype(int))

# Rename columns
seaborne_clean = seaborne_clean.rename(columns={"Economy_Label": "Economy","CargoType_Label": "Cargo_type"})

# Arrange columns
seaborne_clean = seaborne_clean[["Year", "Economy", "Cargo_type", "Trade_thousand_tonnes"]]

# Convert trade values to numeric
seaborne_clean["Trade_thousand_tonnes"] = pd.to_numeric(seaborne_clean["Trade_thousand_tonnes"],errors="coerce")

print("Standardized columns:")
print(seaborne_clean.columns.tolist())
seaborne_clean.head()

Standardized columns:
['Year', 'Economy', 'Cargo_type', 'Trade_thousand_tonnes']


,Year,Economy,Cargo_type,Trade_thousand_tonnes
0,2000,World,Crude oil loaded,1812768.0
1,2000,World,Other tanker trade loaded,1171172.0
2,2000,World,Dry cargo loaded,3361790.0
3,2000,World,Total goods loaded,6345730.0
4,2000,World,Crude oil discharged,1812768.0


In [179]:
# Clean categorical values
# Preserve official names and meaningful differences between economy groups.
categorical_columns = ["Economy", "Cargo_type"]

for column in categorical_columns:
    seaborne_clean[column] = (seaborne_clean[column].astype("string").str.normalize("NFC").str.replace("\u00a0", " ", regex=False).str.replace(r"\s+", " ", regex=True).str.strip().replace("", pd.NA))

In [180]:
# Convert numeric columns and handle non-finite values
seaborne_clean["Year"] = pd.to_numeric(seaborne_clean["Year"], errors="coerce")

seaborne_clean["Trade_thousand_tonnes"] = pd.to_numeric(seaborne_clean["Trade_thousand_tonnes"], errors="coerce")

seaborne_clean = seaborne_clean.replace([np.inf, -np.inf], np.nan)

print("Missing values before removal:")
seaborne_clean.isna().sum()

Missing values before removal:


Year                       0
Economy                    0
Cargo_type                 0
Trade_thousand_tonnes    241
dtype: int64

In [181]:
# Remove rows containing missing values
# Do not replace missing trade values with zero.

rows_before = len(seaborne_clean)

seaborne_clean = seaborne_clean.dropna(subset=["Year", "Economy", "Cargo_type", "Trade_thousand_tonnes"]).copy()

print("Rows removed:", rows_before - len(seaborne_clean))

Rows removed: 241


In [182]:
# Validate years and trade values
invalid_rows = (
    ~seaborne_clean["Year"].between(2000, 2024)
    | seaborne_clean["Year"].mod(1).ne(0)
    | seaborne_clean["Trade_thousand_tonnes"].lt(0)
)

print("Invalid rows:", invalid_rows.sum())

seaborne_clean = seaborne_clean.loc[~invalid_rows].copy()

# Zero trade values are valid and are retained.
seaborne_clean["Year"] = seaborne_clean["Year"].astype("int16")

seaborne_clean["Trade_thousand_tonnes"] = (seaborne_clean["Trade_thousand_tonnes"].astype("float64"))

Invalid rows: 0


In [183]:
# Remove duplicate records
key_columns = ["Year", "Economy", "Cargo_type"]

rows_before = len(seaborne_clean)

# Check whether duplicate keys have conflicting trade values
conflicts = (seaborne_clean.groupby(key_columns)["Trade_thousand_tonnes"].nunique())

if conflicts.gt(1).any():
    print("Warning: Duplicate keys have different trade values.")
    print(seaborne_clean[seaborne_clean.duplicated(key_columns, keep=False)].sort_values(key_columns))
else:
    seaborne_clean = seaborne_clean.drop_duplicates(subset=key_columns,keep="first").copy()

    print("Duplicate rows removed:", rows_before - len(seaborne_clean))

Duplicate rows removed: 0


In [184]:
# Detect potential outliers within each economy and cargo type
# A global Z-score would mix economies of different sizes.
# Use group-specific Z-scores for review only.

grouped_values = seaborne_clean.groupby(["Economy", "Cargo_type"])["Trade_thousand_tonnes"]

group_mean = grouped_values.transform("mean")
group_std = grouped_values.transform("std").replace(0, np.nan)

z_scores = (seaborne_clean["Trade_thousand_tonnes"] - group_mean) / group_std

potential_outliers = (seaborne_clean.loc[z_scores.abs().gt(3)].assign(z_score=z_scores.loc[z_scores.abs().gt(3)]).sort_values("z_score",key=lambda values: values.abs(),ascending=False))

print("Potential outliers requiring review:", len(potential_outliers))
print(potential_outliers.to_string(index=False))

Potential outliers requiring review: 6
 Year                                           Economy                Cargo_type  Trade_thousand_tonnes  z_score
 2024 SIDS (Small island developing States) (UN-OHRLLS)          Crude oil loaded                33922.0 3.721186
 2008                           LDCs: Islands and Haiti      Crude oil discharged                   75.0 3.701860
 2024                                   SIDS: Caribbean          Crude oil loaded                32264.0 3.701154
 2023                                        LDCs: Asia      Crude oil discharged                 1948.0 3.130640
 2008                                            Africa Other tanker trade loaded               142454.0 3.113817
 2008                      Developing economies: Africa Other tanker trade loaded               142412.0 3.113583


In [185]:
# Finalize categorical data types and sort
for column in categorical_columns:
    seaborne_clean[column] = seaborne_clean[column].astype("category")

seaborne_clean = (seaborne_clean.sort_values(["Economy", "Cargo_type", "Year"]).reset_index(drop=True))

In [186]:
# Validate the cleaned dataset
key_columns = ["Year", "Economy", "Cargo_type"]

assert seaborne_clean.notna().all().all()
assert not seaborne_clean.duplicated(subset=key_columns).any()
assert seaborne_clean["Year"].between(2000, 2024).all()
assert seaborne_clean["Trade_thousand_tonnes"].ge(0).all()
assert np.isfinite(seaborne_clean["Trade_thousand_tonnes"]).all()

print("Final shape:", seaborne_clean.shape)

print("\nData types:")
print(seaborne_clean.dtypes)

print("\nMissing values:")
print(seaborne_clean.isna().sum())

Final shape: (8084, 4)

Data types:
Year                        int16
Economy                  category
Cargo_type               category
Trade_thousand_tonnes     float64
dtype: object

Missing values:
Year                     0
Economy                  0
Cargo_type               0
Trade_thousand_tonnes    0
dtype: int64


In [187]:
# Save cleaned dataset in processed folder
seaborne_clean.to_csv("Datasets/processed/SeaborneTrade_Cleaned.csv",index=False)

print("Saved: Datasets/processed/SeaborneTrade_Cleaned.csv")

Saved: Datasets/processed/SeaborneTrade_Cleaned.csv


Raw file: 333 rows (37 economies × 9 cargo types) with one wide column per year, 2000–2024. The file has no footnotes.
Reshaped from wide to long so each row is one economy, cargo type and year (8,325 rows); columns renamed to `Year`, `Economy`, `Cargo_type` and `Trade_thousand_tonnes`.
Text cleaned (spaces and hidden characters) without changing the official names, since groupings such as "LDCs: Asia" and "SIDS: Caribbean" differ in meaning.
Year and trade values set to numbers; infinite values would be treated as missing. 241 values are missing.
The 241 missing rows are removed, never filled with zero. 225 belong to "Individual economies", a placeholder row marked "Not applicable", and the other 16 are in two island groupings (LDCs: Islands and Haiti and SIDS: Atlantic and Indian Ocean).
Years run only from 2000 to 2024 and no value is negative. Zero trade is a valid value and is kept.
No duplicate economy-year-cargo records, so nothing is dropped.
Outliers are checked within each economy and cargo type (|z| above 3), not globally, so large and small economies are not mixed: 6 values flagged, all in island or small groupings, such as crude oil loaded by the SIDS group in 2024. They look like real changes, so they are kept.
Economy and cargo type set to categories and the data sorted.
All checks pass: no missing values, no duplicates, years 2000–2024, no negative values. Final table: 8,084 rows, 36 economies.
Saved as `SeaborneTrade_Cleaned.csv`.

#### US.MerchantFleet

In [188]:
# merchant fleet
fleet.isnull().sum()

Year                                                0
Economy                                             0
Economy Label                                       0
ShipType                                            0
ShipType Label                                      0
Dead weight tons in thousands                    5082
Dead weight tons in thousands Footnote          63872
Dead weight tons in thousands Missing value     58814
Percentage of total world                        5082
Percentage of total world Footnote              63872
Percentage of total world Missing value         58814
Percentage of total fleet                        5082
Percentage of total fleet Footnote              63872
Percentage of total fleet Missing value         58814
Number of ships                                 39296
Number of ships Footnote                        63884
Number of ships Missing value                   24600
Gross Tonnage in thousands                      39296
Gross Tonnage in thousands F

In [189]:
# Inspect
fleet_raw = fleet.copy()
print("Shape:", fleet_raw.shape)
print("Years:", fleet_raw["Year"].min(), "-", fleet_raw["Year"].max())
print("Economies:", fleet_raw["Economy"].nunique(), "| ship types:", fleet_raw["ShipType Label"].unique().tolist())
print("Duplicate economy-year-type records:", fleet_raw.duplicated(["Year", "Economy", "ShipType"]).sum())
print(fleet_raw.isnull().sum().to_string())

Shape: (63896, 23)
Years: 1980 - 2026
Economies: 291 | ship types: ['Total fleet', 'Oil tankers', 'Bulk carriers', 'General cargo', 'Container ships', 'Other types of ships']
Duplicate economy-year-type records: 0
Year                                                0
Economy                                             0
Economy Label                                       0
ShipType                                            0
ShipType Label                                      0
Dead weight tons in thousands                    5082
Dead weight tons in thousands Footnote          63872
Dead weight tons in thousands Missing value     58814
Percentage of total world                        5082
Percentage of total world Footnote              63872
Percentage of total world Missing value         58814
Percentage of total fleet                        5082
Percentage of total fleet Footnote              63872
Percentage of total fleet Missing value         58814
Number of ships               

In [190]:
# Short column names
MEASURES = {"DWT_k": "Dead weight tons in thousands", "Share_world": "Percentage of total world",
            "Share_fleet": "Percentage of total fleet", "Ships": "Number of ships",
            "GT_k": "Gross Tonnage in thousands", "Age": "Average age of vessels (years)"}

rename = {"Economy": "Code", "Economy Label": "Economy", "ShipType": "ShipTypeCode", "ShipType Label": "ShipType"}
for short, long in MEASURES.items():
    rename[long] = short
    rename[long + " Footnote"] = short + "_footnote"
    rename[long + " Missing value"] = short + "_missing"

fleet = fleet_raw.rename(columns=rename)
print(fleet[list(MEASURES)].dtypes.to_string())

DWT_k          float64
Share_world    float64
Share_fleet    float64
Ships          float64
GT_k           float64
Age            float64


In [191]:
# Series breaks (the only footnote in the file)
for m in MEASURES:
    fleet["break_" + m] = fleet[m + "_footnote"].eq("Break")
fleet["series_break"] = fleet[["break_" + m for m in MEASURES]].any(axis=1)
print(fleet[fleet.series_break].groupby("Economy").Year.unique().to_string())

Economy
France                  [2011]
United Kingdom          [1995]
United States     [2002, 2011]


In [192]:
# Missing values: every gap has a reason
for m in MEASURES:
    v, r = fleet[m], fleet[m + "_missing"]
    print(f"{m:12} value with a reason: {(v.notna() & r.notna()).sum()} | empty without a reason: {(v.isna() & r.isna()).sum()} | {r.value_counts().to_dict()}")

tf = fleet[fleet.ShipType == "Total fleet"]
print(tf.groupby("Year")[["DWT_k", "Ships", "GT_k", "Age"]].count().loc[[1980, 2010, 2011, 2026]])

DWT_k        value with a reason: 0 | empty without a reason: 0 | {'Not applicable': 5082}
Share_world  value with a reason: 0 | empty without a reason: 0 | {'Not applicable': 5082}
Share_fleet  value with a reason: 0 | empty without a reason: 0 | {'Not applicable': 5082}
Ships        value with a reason: 0 | empty without a reason: 0 | {'No value reported or collected': 34214, 'Not applicable': 5082}
GT_k         value with a reason: 0 | empty without a reason: 0 | {'No value reported or collected': 34214, 'Not applicable': 5082}
Age          value with a reason: 0 | empty without a reason: 0 | {'No value reported or collected': 38791, 'Not applicable': 5082}
      DWT_k  Ships  GT_k  Age
Year                         
1980    227      0     0    0
2010    243      0     0    0
2011    251    251   251  251
2026    267    267   267  267


In [193]:
# Economy level
def level(code):
    try:
        c = int(code)
    except ValueError:
        return "unallocated"
    if c == 0:
        return "world"
    return "grouping" if c == 1 or c >= 1000 else "country"

fleet["level"] = fleet["Code"].map(level)
print(fleet.groupby("level").Economy.nunique().to_dict())

{'country': 221, 'grouping': 68, 'unallocated': 1, 'world': 1}


In [194]:
#%% Step 6. ISO3 for the country rows
# West Germany (280) continues as Germany (276) from 1990, so it takes DEU
CODE_FIX = {280: "DEU"}

def to_iso3(code):
    c = int(code)
    if c in CODE_FIX:
        return CODE_FIX[c]
    hit = pycountry.countries.get(numeric=str(c).zfill(3))
    return hit.alpha_3 if hit else None

cn = fleet.loc[fleet.level == "country", ["Code", "Economy"]].drop_duplicates()
iso = {c: to_iso3(c) for c in cn.Code}
fleet["ISO3"] = [iso.get(c) if l == "country" else None for c, l in zip(fleet.Code, fleet.level)]
former = sorted(cn[cn.Code.map(iso).isna()].Economy)
fleet["former_state"] = fleet.level.eq("country") & fleet.ISO3.isna()
print("countries:", len(cn), "| without ISO3 (dissolved states):", len(former))
print(former)

countries: 221 | without ISO3 (dissolved states): 12
['Czechoslovakia', 'Democratic Republic of Germany', 'Ethiopia (...1991)', 'Indonesia (...2002)', 'Netherlands Antilles', 'Panama, excluding Canal Zone', 'Serbia and Montenegro', 'Sudan (...2011)', 'Union of Soviet Socialist Republics', 'Yemen, Arab Republic', 'Yemen, Democratic', 'Yugoslavia, Soc. Fed. Rep. of']


In [195]:
# Average age: flag, do not delete
fleet["Age_raw"] = fleet["Age"]
fleet["age_odd"] = (fleet.Age <= 0) | (fleet.Age > 60)
fleet["age_small_fleet"] = fleet.Age.notna() & fleet.Ships.le(5)
odd = fleet[fleet.age_odd]
print("odd ages:", len(odd), "| in fleets of 5 ships or fewer:", int(odd.Ships.le(5).sum()), "| levels:", odd.level.value_counts().to_dict())
print(odd.sort_values("Age", ascending=False)[["Year", "Economy", "ShipType", "Ships", "Age"]].head(3).to_string(index=False))
print("age rows flagged small fleet:", int(fleet.age_small_fleet.sum()), "of", int(fleet.Age.notna().sum()))

odd ages: 216 | in fleets of 5 ships or fewer: 203 | levels: {'country': 216}
 Year                  Economy      ShipType  Ships    Age
 2026 Turks and Caicos Islands General cargo    1.0 109.92
 2025 Turks and Caicos Islands General cargo    1.0 108.92
 2024 Turks and Caicos Islands General cargo    1.0 107.92
age rows flagged small fleet: 3242 of 20023


In [196]:
# Ship types add up to the total, one measure at a time
rows, failures = [], []
total_fleet = fleet[fleet.ShipType == "Total fleet"]
parts_fleet = fleet[fleet.ShipType != "Total fleet"]

for m in ("DWT_k", "GT_k", "Ships"):
    total = total_fleet.set_index(["Year", "Code"])[m]
    parts = parts_fleet.groupby(["Year", "Code"])[m].agg(sum=lambda s: s.sum(min_count=1), count="count")
    x = pd.concat([total.rename("total"), parts], axis=1).dropna(subset=["total", "sum"])
    x = x[x.total > 0]
    x["gap_pct"] = (x["sum"] / x.total - 1) * 100
    failures.append(x[x.gap_pct.abs() >= 1].assign(measure=m).reset_index())

    c = total_fleet[total_fleet.level == "country"].groupby("Year")[m].sum(min_count=1)
    w = total_fleet[total_fleet.level == "world"].set_index("Year")[m]
    ratio = (c / w.where(w > 0)).dropna()
    rows.append({"measure": m, "checked": len(x), "all 5 types present": round((x["count"] == 5).mean(), 3),
                 "within 1%": round((x.gap_pct.abs() < 1).mean(), 4), "failures": int((x.gap_pct.abs() >= 1).sum()),
                 "countries / World min": round(ratio.min(), 3), "latest": round(ratio.iloc[-1], 3)})

print(pd.DataFrame(rows).set_index("measure").to_string())
failures = pd.concat(failures, ignore_index=True)
failures.merge(fleet[["Code", "Economy"]].drop_duplicates(), on="Code").round(2)

         checked  all 5 types present  within 1%  failures  countries / World min  latest
measure                                                                                  
DWT_k      11115                0.686     0.9992         9                  0.991   0.991
GT_k        4100                1.000     1.0000         0                  0.992   0.992
Ships       4100                1.000     1.0000         0                  0.979   0.980


,Year,Code,total,sum,count,gap_pct,measure,Economy
0,1990,660,4.00,3.00,1,-25.00,DWT_k,Anguilla
1,1990,230,94.00,93.00,2,-1.06,DWT_k,Ethiopia (...1991)
2,1990,520,45.00,47.00,3,4.44,DWT_k,Nauru
3,1990,768,65.00,66.00,3,1.54,DWT_k,Togo
4,1991,768,78.00,79.00,3,1.28,DWT_k,Togo
5,2006,5420,63203.05,65369.28,5,3.43,DWT_k,Northern Europe
6,2006,826,11662.39,13828.61,5,18.57,DWT_k,United Kingdom
7,2006,3916,165288.65,167454.88,5,1.31,DWT_k,G20 (Group of Twenty)
8,2006,3905,212284.27,214450.50,5,1.02,DWT_k,OECD (Organisation for Economic Cooperation an...


In [197]:
# Analytical subset and coverage
LATEST = int(fleet.Year.max())
fleet["latest_year_flag"] = fleet.Year.eq(LATEST)

fleet_country = fleet[(fleet.level == "country")
                      & fleet.ISO3.notna()
                      & ~fleet.DWT_k_missing.eq("Not applicable")].copy()

cov = (fleet_country[fleet_country.ShipType == "Total fleet"].groupby("Year").DWT_k.sum(min_count=1)
       / fleet[(fleet.level == "world") & (fleet.ShipType == "Total fleet")].set_index("Year").DWT_k)
fleet_country["coverage_world"] = fleet_country.Year.map(cov)

print("latest year:", LATEST, "| flagged rows:", int(fleet.latest_year_flag.sum()))
print("coverage before 1992: %.3f-%.3f | from 1992: %.3f-%.3f" % (cov[cov.index < 1992].min(), cov[cov.index < 1992].max(), cov[cov.index >= 1992].min(), cov[cov.index >= 1992].max()))
print("subset:", fleet_country.shape, "| countries:", fleet_country.ISO3.nunique(),
      "| Ships, GT and Age from", int(fleet_country.dropna(subset=["Ships"]).Year.min()))

latest year: 2026 | flagged rows: 1680
coverage before 1992: 0.897-0.950 | from 1992: 0.991-1.000
subset: (39095, 38) | countries: 208 | Ships, GT and Age from 2011


In [198]:
# Validate and save
assert not fleet.duplicated(["Year", "Code", "ShipType"]).any()
assert not fleet_country.duplicated(["ISO3", "Year", "ShipType"]).any()
assert fleet_country.DWT_k.notna().all()

fleet.to_csv(PROC / "fleet_clean.csv", index=False)
fleet_country.to_csv(PROC / "fleet_country.csv", index=False)
print("saved:", fleet.shape, fleet_country.shape)

saved: (63896, 37) (39095, 38)


Raw file: 63,896 rows, 291 economies, six ship types, 1980–2026, no duplicates.
Columns renamed to short names; footnote and missing-value columns kept.
The only footnote is "Break" (24 rows: France, UK, US); flagged and kept.
Every gap has a reason; ships, tonnage and age exist only from 2011, dead weight from 1980.
Economies split into 221 countries, 68 groupings, world and unallocated.
ISO3 added from UN codes; West Germany maps to Germany; 12 dissolved states have none and are flagged.
216 odd ages are mostly one-ship fleets, so they are kept and flagged, not removed.
Ship types add up to the total within 1% except 9 dead-weight rows (small-fleet rounding and the UK in 2006).
Country subset: 39,095 rows, 208 countries; covers 90–95% of the world fleet before 1992 and 99–100% after.
No duplicates; both tables saved.

#### LSCI

In [199]:
# LSCI
lsci.isnull().sum()

Economy_Label                                  0
Quarter_Label                                  0
Index_average_Q1_2023__100_Value             289
Index_average_Q1_2023__100_Footnote        15013
Index_average_Q1_2023__100_MissingValue    14724
dtype: int64

In [200]:
lsci.dtypes

Economy_Label                               object
Quarter_Label                               object
Index_average_Q1_2023__100_Value           float64
Index_average_Q1_2023__100_Footnote        float64
Index_average_Q1_2023__100_MissingValue     object
dtype: object

In [201]:
# Inspect
lsci = pd.read_csv("Datasets/raw/US.LSCI_20261121_091634.csv")
print("Shape:", lsci.shape)
print(lsci.dtypes.to_string())
print(lsci.isnull().sum().to_string())
print("Duplicate economy-quarter records:", lsci.duplicated(["Economy_Label", "Quarter_Label"]).sum())

Shape: (15013, 5)
Economy_Label                               object
Quarter_Label                               object
Index_average_Q1_2023__100_Value           float64
Index_average_Q1_2023__100_Footnote        float64
Index_average_Q1_2023__100_MissingValue     object
Economy_Label                                  0
Quarter_Label                                  0
Index_average_Q1_2023__100_Value             289
Index_average_Q1_2023__100_Footnote        15013
Index_average_Q1_2023__100_MissingValue    14724
Duplicate economy-quarter records: 0


In [202]:
# Short column names
lsci_clean = lsci.rename(columns={
    "Economy_Label": "Economy", "Quarter_Label": "Quarter",
    "Index_average_Q1_2023__100_Value": "LSCI",
    "Index_average_Q1_2023__100_MissingValue": "LSCI_missing"}).copy()

print("footnotes present:", int(lsci_clean["Index_average_Q1_2023__100_Footnote"].notna().sum()))
lsci_clean = lsci_clean.drop(columns=["Index_average_Q1_2023__100_Footnote"])
lsci_clean["Economy"] = lsci_clean["Economy"].astype("string").str.strip()

footnotes present: 0


In [203]:
# Quarter, year and a sortable date
print("quarter labels in the form Qn yyyy:", bool(lsci_clean.Quarter.str.match(r"^Q[1-4] \d{4}$").all()))
lsci_clean["Year"] = lsci_clean.Quarter.str[-4:].astype(int)
lsci_clean["Quarter_Number"] = lsci_clean.Quarter.str[1].astype(int)
lsci_clean["Date"] = pd.to_datetime({"year": lsci_clean.Year, "month": (lsci_clean.Quarter_Number - 1) * 3 + 1, "day": 1})

lsci_clean = lsci_clean.sort_values(["Economy", "Date"]).reset_index(drop=True)
print("period:", lsci_clean.Quarter[lsci_clean.Date.idxmin()], "-", lsci_clean.Quarter[lsci_clean.Date.idxmax()], "| quarters:", lsci_clean.Date.nunique())

quarter labels in the form Qn yyyy: True
period: Q1 2006 - Q3 2026 | quarters: 83


In [204]:
# Panel coverage
n_econ, n_q = lsci_clean.Economy.nunique(), lsci_clean.Date.nunique()
per = lsci_clean.groupby("Economy").Date.nunique()
print("economies:", n_econ, "| rows if complete:", n_econ * n_q, "| actual rows:", len(lsci_clean), "| economy-quarters with no row:", n_econ * n_q - len(lsci_clean))
print("quarters per economy:", per.value_counts().sort_index().to_dict())

economies: 191 | rows if complete: 15853 | actual rows: 15013 | economy-quarters with no row: 840
quarters per economy: {7: 1, 8: 1, 11: 1, 13: 1, 28: 1, 36: 2, 37: 1, 38: 1, 42: 1, 43: 3, 45: 1, 60: 1, 67: 2, 73: 1, 75: 1, 76: 3, 79: 1, 80: 2, 81: 2, 83: 164}


In [205]:
# Missing values
print(lsci_clean.LSCI_missing.value_counts(dropna=False).to_string())
print("empty without a reason:", int((lsci_clean.LSCI.isna() & lsci_clean.LSCI_missing.isna()).sum()), "| value with a reason:", int((lsci_clean.LSCI.notna() & lsci_clean.LSCI_missing.notna()).sum()))
gaps = lsci_clean[lsci_clean.LSCI.isna()].groupby("Economy").agg(rows=("Quarter", "size"), first=("Year", "min"), last=("Year", "max"))
print(gaps.sort_values("rows", ascending=False).head(6).to_string())

LSCI_missing
NaN               14724
Not applicable      289
empty without a reason: 0 | value with a reason: 0
                                  rows  first  last
Economy                                            
Serbia and Montenegro               75   2008  2026
Netherlands Antilles                63   2011  2026
Sudan (...2011)                     59   2012  2026
Sudan                               24   2006  2011
Bonaire, Sint Eustatius and Saba    20   2006  2010
Curacao                             20   2006  2010


In [206]:
# ISO3
ALIAS = {"China, Hong Kong SAR": "HKG", "China, Taiwan Province of": "TWN", "Cote d'Ivoire": "CIV", "Curacao": "CUW",
         "Dem. People's Rep. of Korea": "PRK", "Dem. Rep. of the Congo": "COD", "Iran (Islamic Republic of)": "IRN",
         "Micronesia (Federated States of)": "FSM", "Netherlands (Kingdom of the)": "NLD", "Republic of Korea": "KOR",
         "Reunion": "REU", "Saint Helena": "SHN", "Turkiye": "TUR", "United States Virgin Islands": "VIR",
         "Venezuela (Bolivarian Rep. of)": "VEN", "Wallis and Futuna Islands": "WLF"}

def to_iso3(name):
    if name in ALIAS:
        return ALIAS[name]
    try:
        return pycountry.countries.lookup(name).alpha_3
    except LookupError:
        return None

names = lsci_clean.Economy.unique()
iso = {n: to_iso3(n) for n in names}
lsci_clean["ISO3"] = lsci_clean.Economy.map(iso)
print("economies without ISO3 (dissolved):", [n for n in names if iso[n] is None])
print("ISO3 shared by two economies:", int(pd.Series(list(iso.values())).dropna().duplicated().sum()))

economies without ISO3 (dissolved): ['Netherlands Antilles', 'Serbia and Montenegro', 'Sudan (...2011)']
ISO3 shared by two economies: 0


In [207]:
# Values and the base quarter
print(lsci_clean.LSCI.describe().round(1).to_dict())
print("zero or negative:", int((lsci_clean.LSCI <= 0).sum()))
print(lsci_clean.nlargest(3, "LSCI")[["Economy", "Quarter", "LSCI"]].to_string(index=False))
print(lsci_clean.nsmallest(3, "LSCI")[["Economy", "Quarter", "LSCI"]].to_string(index=False))

base = lsci_clean[lsci_clean.Quarter == "Q1 2023"].LSCI
print("base quarter Q1 2023: economies", int(base.count()), "| mean", round(base.mean(), 1), "| median", round(base.median(), 1))

lsci_clean["change"] = lsci_clean.groupby("Economy").LSCI.pct_change()
print("quarter-on-quarter changes above 100%:", int((lsci_clean.change.abs() > 1).sum()), "| economies:", lsci_clean[lsci_clean.change.abs() > 1].Economy.nunique())
print("largest economies' index in the latest quarter:", lsci_clean[lsci_clean.Quarter == "Q3 2026"].nlargest(3, "LSCI")[["Economy", "LSCI"]].values.tolist())

{'count': 14724.0, 'mean': 96.9, 'std': 133.8, 'min': 2.5, '25%': 22.8, '50%': 47.1, '75%': 106.9, 'max': 1342.0}
zero or negative: 0
Economy Quarter    LSCI
  China Q3 2026 1341.97
  China Q1 2026 1322.63
  China Q2 2026 1319.46
       Economy Quarter  LSCI
Norfolk Island Q4 2018   2.5
Norfolk Island Q1 2019   2.5
Norfolk Island Q2 2019   2.5
base quarter Q1 2023: economies 176 | mean 100.5 | median 48.0
quarter-on-quarter changes above 100%: 15 | economies: 13
largest economies' index in the latest quarter: [['China', 1341.97], ['Republic of Korea', 634.62], ['Singapore', 625.67]]


In [208]:
# Economies that appear or disappear
lsci_clean["quarters_available"] = lsci_clean.groupby("Economy").LSCI.transform("count")
cover = lsci_clean.dropna(subset=["LSCI"]).groupby("Economy").agg(first=("Date", "min"), last=("Date", "max"), n=("LSCI", "size"))
print("economies with all", n_q, "quarters:", int((cover.n == n_q).sum()), "| with fewer:", int((cover.n < n_q).sum()))
print(cover[cover.n < n_q].sort_values("n").head(6).assign(first=lambda d: d["first"].dt.date, last=lambda d: d["last"].dt.date).to_string())

economies with all 83 quarters: 157 | with fewer: 34
                                  first        last   n
Economy                                                
Cocos (Keeling) Islands      2006-01-01  2007-07-01   7
Falkland Islands (Malvinas)  2009-04-01  2011-01-01   8
Serbia and Montenegro        2006-01-01  2007-10-01   8
Saint Helena                 2024-01-01  2026-07-01  11
Saint Pierre and Miquelon    2023-07-01  2026-07-01  13
Netherlands Antilles         2006-01-01  2010-10-01  20


In [209]:
# Validate and save
assert not lsci_clean.duplicated(["Economy", "Quarter"]).any()
assert lsci_clean.LSCI.dropna().gt(0).all()

lsci_clean = lsci_clean.drop(columns="change")
lsci_clean.to_csv(PROC / "LSCI_quarterly_cleaned.csv", index=False)
print("saved:", lsci_clean.shape)

saved: (15013, 9)


Raw file: 15,013 rows, one index value per economy and quarter, no duplicates.
Columns renamed; the footnote column is completely empty, so it is dropped.
Quarter labels converted to year, quarter number and a real date, and the data sorted. It runs from Q1 2006 to Q3 2026 (83 quarters).
191 economies. A complete panel would have 15,853 rows, so 840 economy-quarters have no row at all (economies that start late or end early).
289 empty values, all "Not applicable" (the economy did not exist or was not covered then); none are set to zero.
ISO3 added by country name with a short list of fixes; three dissolved economies (Netherlands Antilles, Serbia and Montenegro, Sudan before 2012) have none.
No zero or negative values. China is highest (1,342 in Q3 2026). The average in the base quarter Q1 2023 is 100.5, so the index base holds.
164 economies have rows for every quarter; 157 also have a value in every quarter. The rest start late or end early.
No duplicates and every value is positive; table saved.

#### Container port throughput

In [210]:
# port throughout
teu.isnull().sum()

Year                                                  0
Economy                                               0
Economy Label                                         0
TEU (Twenty foot Equivalent Unit)                   130
TEU (Twenty foot Equivalent Unit) Footnote         3011
TEU (Twenty foot Equivalent Unit) Missing value    2900
dtype: int64

In [211]:
# Inspect
print("Shape:", teu.shape)
print(teu.dtypes.to_string())
print("Years:", teu["Year"].min(), "-", teu["Year"].max(), "| economies:", teu["Economy"].nunique())
print("Duplicate economy-year records:", teu.duplicated(["Year", "Economy"]).sum())
print(teu.isnull().sum().to_string())

Shape: (3030, 6)
Year                                                 int64
Economy                                              int64
Economy Label                                       object
TEU (Twenty foot Equivalent Unit)                  float64
TEU (Twenty foot Equivalent Unit) Footnote          object
TEU (Twenty foot Equivalent Unit) Missing value     object
Years: 2010 - 2024 | economies: 202
Duplicate economy-year records: 0
Year                                                  0
Economy                                               0
Economy Label                                         0
TEU (Twenty foot Equivalent Unit)                   130
TEU (Twenty foot Equivalent Unit) Footnote         3011
TEU (Twenty foot Equivalent Unit) Missing value    2900


In [212]:
# Short column names
teu_clean = teu.rename(columns={
    "Economy": "Code", "Economy Label": "Economy",
    "TEU (Twenty foot Equivalent Unit)": "TEU",
    "TEU (Twenty foot Equivalent Unit) Footnote": "TEU_footnote",
    "TEU (Twenty foot Equivalent Unit) Missing value": "TEU_missing"}).copy()
teu_clean["Economy"] = teu_clean["Economy"].astype("string").str.strip()

In [213]:
# Economy level
def level(code):
    c = int(code)
    return "world" if c == 0 else ("grouping" if c >= 1000 else "country")

teu_clean["level"] = teu_clean["Code"].map(level)
print(teu_clean.groupby("level").Code.nunique().to_dict())
print(teu_clean[teu_clean.level == "grouping"].Economy.unique().tolist())

{'country': 136, 'grouping': 65, 'world': 1}
['Africa', 'Northern Africa', 'Sub-Saharan Africa', 'Eastern Africa', 'Middle Africa', 'Southern Africa', 'Western Africa', 'Americas', 'Northern America', 'Latin America and the Caribbean', 'Caribbean', 'Central America', 'South America', 'Asia', 'Eastern Asia', 'South-eastern Asia', 'Southern Asia', 'Western Asia', 'Europe', 'Eastern Europe', 'Northern Europe', 'Southern Europe', 'Western Europe', 'Oceania', 'Australia and New Zealand', 'Oceania excluding Australia and New Zealand', 'Asia and Oceania', 'Northern America and Europe', 'Europe, Northern America, Australia and New Zealand', 'Central and Southern Asia', 'Eastern and South-Eastern Asia', 'Western Asia and Northern Africa', 'Developing economies', 'Developing economies: Africa', 'Developing economies: Northern Africa', 'Developing economies: Sub-Saharan Africa', 'Developing economies: Americas', 'Developing economies: Asia and Oceania', 'Developing economies: Asia', 'Developing e

In [214]:
# ISO3 for the country rows
def to_iso3(code):
    hit = pycountry.countries.get(numeric=str(int(code)).zfill(3))
    return hit.alpha_3 if hit else None

cn = teu_clean.loc[teu_clean.level == "country", ["Code", "Economy"]].drop_duplicates()
teu_clean["ISO3"] = teu_clean["Code"].map({c: to_iso3(c) for c in cn.Code}).where(teu_clean.level == "country")
print("countries:", len(cn), "| without ISO3:", cn[cn.Code.map(to_iso3).isna()].Economy.tolist())

countries: 136 | without ISO3: ['Netherlands Antilles', 'Sudan (...2011)']


In [215]:
# Missing values and footnotes
print(teu_clean.TEU_missing.value_counts(dropna=False).to_string())
print(teu_clean.TEU_footnote.value_counts(dropna=False).to_string())
print("empty without a reason:", int((teu_clean.TEU.isna() & teu_clean.TEU_missing.isna()).sum()), "| value with a reason:", int((teu_clean.TEU.notna() & teu_clean.TEU_missing.notna()).sum()))
print(teu_clean[teu_clean.TEU.isna()].groupby(["Economy", "TEU_missing"]).Year.agg(["min", "max", "count"]).to_string())

TEU_missing
NaN                2900
Not publishable      99
Not applicable       31
TEU_footnote
NaN                                                          3011
Estimated                                                      15
Revised more than 20% in relation to the previous update.       4
empty without a reason: 0 | value with a reason: 0
                                                 min   max  count
Economy                        TEU_missing                       
Albania                        Not publishable  2024  2024      1
Algeria                        Not publishable  2021  2021      1
Aruba                          Not publishable  2011  2012      2
Barbados                       Not publishable  2024  2024      1
Belize                         Not publishable  2024  2024      1
Brunei Darussalam              Not publishable  2024  2024      1
Cayman Islands                 Not publishable  2018  2023      5
Cuba                           Not publishable  2024  2024  

In [216]:
# Panel: every economy should have every year
per = teu_clean.groupby("Code").Year.nunique()
print("years per economy:", per.value_counts().to_dict())
print("years:", sorted(teu_clean.Year.unique()))

years per economy: {15: 202}
years: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


In [217]:
# Values
print("zero or negative:", int((teu_clean.TEU <= 0).sum()), "| range:", teu_clean.TEU.min(), "-", teu_clean.TEU.max())
print(teu_clean[teu_clean.level == "country"].nlargest(3, "TEU")[["Year", "Economy", "TEU"]].to_string(index=False))

c = teu_clean[teu_clean.level == "country"].sort_values(["Code", "Year"]).copy()
c["change"] = c.groupby("Code").TEU.pct_change()
print("year-on-year changes above 100%:")
print(c[c.change.abs() > 1][["Year", "Economy", "TEU", "change"]].round(2).to_string(index=False))

zero or negative: 0 | range: 9500.0 - 920300805.0
 Year Economy         TEU
 2024   China 299703800.0
 2023   China 278759834.0
 2022   China 268790000.0
year-on-year changes above 100%:
 Year    Economy      TEU  change
 2019      Libya 265098.0    1.51
 2021 Montenegro  38352.0    1.44
 2015       Togo 905700.0    1.38


In [218]:
# Countries against the World total
world = teu_clean[teu_clean.level == "world"].set_index("Year").TEU
ctry = teu_clean[teu_clean.level == "country"].groupby("Year").TEU.sum(min_count=1)
cov = (ctry / world).round(3)
print("countries / World:", cov.min(), "-", cov.max())
print(cov.to_dict())

countries / World: 0.993 - 1.0
{2010: 0.999, 2011: 0.999, 2012: 1.0, 2013: 1.0, 2014: 1.0, 2015: 0.999, 2016: 0.999, 2017: 0.999, 2018: 1.0, 2019: 0.998, 2020: 0.998, 2021: 0.995, 2022: 0.995, 2023: 0.993, 2024: 0.993}


In [219]:
# Country table and save
teu_country = teu_clean[teu_clean.level == "country"].drop(columns="level").sort_values(["Code", "Year"]).reset_index(drop=True)
assert not teu_country.duplicated(["Code", "Year"]).any()
assert teu_country.TEU.dropna().gt(0).all()

teu_clean.to_csv(PROC / "TEU_clean.csv", index=False)
teu_country.to_csv(PROC / "TEU_country_cleaned.csv", index=False)
print("saved:", teu_clean.shape, teu_country.shape)

saved: (3030, 8) (2040, 7)


Raw file: 3,030 rows, 202 economies, 2010–2024, no duplicates.
Columns renamed and a working copy made.
Economies split into 136 countries, 65 groupings and the world total.
ISO3 added from numeric codes; only two former countries have none.
130 empty values, all with a reason; left empty, never set to zero.
Every economy has all 15 years.
No zero or negative values; the only jump above 100% (Togo 2015) looks real and is kept.
Countries make up 99.3–100% of the World total every year.
Full table and country-only table saved.

#### CPPI

In [220]:
cppi.isnull().sum()

Port                                2
UN/LOCODE                           7
Territory                           7
World Bank Region                   7
Region average CPPI                 7
World Bank Income Group             7
Income Group average CPPI           7
Maritime services region            7
Region's average cppi               7
CPPI 2020                         100
CPPI 2021                          74
CPPI 2022                          86
CPPI 2023                          35
CPPI 2024                          23
CPPI 2025                          26
Rank 2025                          26
Berth hours in % of port hours     26
Statistical Index                  26
Administrative Index               26
Number of calls in sample          26
dtype: int64

In [221]:
# Inspect
print("Shape:", cppi.shape)
print("Fully duplicated rows:", cppi.duplicated().sum())
print("Rows without a UN/LOCODE:", int(cppi["UN/LOCODE"].isna().sum()))
print(cppi.loc[cppi["UN/LOCODE"].isna(), "Port"].dropna().str[:60].to_string())

Shape: (426, 20)
Fully duplicated rows: 1
Rows without a UN/LOCODE: 7
421    * Yangshan has been incorporated into Shanghai...
422    ** Venezuela, RB classified as an upper-middle...
423    a World Bank Region: EAP = East Asia and Pacif...
424    b World Bank Income Groups: HI = High Income, ...
425    c The share of time at berth in per cent of to...


In [222]:
# Keep port rows only (blank lines and footnotes have no UN/LOCODE)
cppi_clean = cppi[cppi["UN/LOCODE"].notna()].copy()
print("Port rows:", len(cppi_clean), "| fully duplicated:", cppi_clean.duplicated().sum())

Port rows: 419 | fully duplicated: 0


In [223]:
# Column names and text
cppi_clean = cppi_clean.rename(columns={
    "UN/LOCODE": "LOCODE", "Territory": "Country",
    "World Bank Region": "WB_region", "Region average CPPI": "WB_region_avg_CPPI",
    "World Bank Income Group": "Income_group", "Income Group average CPPI": "Income_group_avg_CPPI",
    "Maritime services region": "Maritime_region", "Region's average cppi": "Maritime_region_avg_CPPI",
    "Berth hours in % of port hours": "Berth_share", "Statistical Index": "Statistical_index",
    "Administrative Index": "Administrative_index", "Number of calls in sample": "Calls_in_sample",
    "Rank 2025": "Rank_2025"})

for col in ["Port", "LOCODE", "Country", "WB_region", "Income_group", "Maritime_region"]:
    cppi_clean[col] = cppi_clean[col].astype("string").str.normalize("NFC").str.replace(r"\s+", " ", regex=True).str.strip()
cppi_clean["Country"] = cppi_clean["Country"].str.replace("\u2019", "'", regex=False)
cppi_clean["LOCODE"] = cppi_clean["LOCODE"].str.upper()

In [224]:
# Keys
print("LOCODE length:", cppi_clean.LOCODE.str.len().value_counts().to_dict())
print("duplicate LOCODE:", cppi_clean.LOCODE.duplicated().sum(), "| duplicate port names:", cppi_clean.Port.duplicated().sum())
print("countries:", cppi_clean.Country.nunique(), "| LOCODE prefixes per country:", int((cppi_clean.groupby("Country").LOCODE.apply(lambda s: s.str[:2].nunique()) > 1).sum()), "with more than one")

LOCODE length: {np.int64(5): 419}
duplicate LOCODE: 0 | duplicate port names: 0
countries: 134 | LOCODE prefixes per country: 0 with more than one


In [225]:
# ISO3 from the LOCODE country prefix
def iso3_from_locode(code):
    hit = pycountry.countries.get(alpha_2=code[:2])
    return hit.alpha_3 if hit else None

cppi_clean["ISO3"] = cppi_clean.LOCODE.map(iso3_from_locode)
print("ports without ISO3:", int(cppi_clean.ISO3.isna().sum()), cppi_clean[cppi_clean.ISO3.isna()].LOCODE.tolist())
print(cppi_clean[cppi_clean.Country.isin(["Canary Islands", "Guam", "Taiwan, China", "Hong Kong SAR, China"])][["Port", "Country", "LOCODE", "ISO3"]].drop_duplicates("Country").to_string(index=False))


ports without ISO3: 0 []
                 Port              Country LOCODE ISO3
          Apra Harbor                 Guam  GUAPR  GUM
Arrecife De Lanzarote       Canary Islands  ESACE  ESP
            Hong Kong Hong Kong SAR, China  HKHKG  HKG
            Kaohsiung        Taiwan, China  TWKHH  TWN


In [226]:
# Scores, years available and missing values
score_cols = [f"CPPI {y}" for y in range(2020, 2026)]
cppi_clean["years_available"] = cppi_clean[score_cols].notna().sum(axis=1)
print(cppi_clean.years_available.value_counts().sort_index().to_dict())
print(cppi_clean[score_cols + ["Rank_2025", "Berth_share", "Statistical_index", "Administrative_index", "Calls_in_sample"]].isna().sum().to_string())

no25 = cppi_clean[cppi_clean["CPPI 2025"].isna()]
print("ports without a 2025 score:", len(no25), "| of which have earlier scores:", int((no25.years_available > 0).sum()))

{1: 14, 2: 16, 3: 31, 4: 20, 5: 35, 6: 303}
CPPI 2020               93
CPPI 2021               67
CPPI 2022               79
CPPI 2023               28
CPPI 2024               16
CPPI 2025               19
Rank_2025               19
Berth_share             19
Statistical_index       19
Administrative_index    19
Calls_in_sample         19
ports without a 2025 score: 19 | of which have earlier scores: 19


In [227]:
# Consistency of the 2025 columns
r = cppi_clean.dropna(subset=["Rank_2025"]).sort_values("Rank_2025")
print("ranks:", int(r.Rank_2025.min()), "-", int(r.Rank_2025.max()), "| duplicated ranks:", int(r.Rank_2025.duplicated().sum()), "| ranks match 2025 scores in order:", bool(r["CPPI 2025"].is_monotonic_decreasing))
print("rank and score missing together:", bool(cppi_clean.Rank_2025.isna().equals(cppi_clean["CPPI 2025"].isna())))
print("berth share range:", cppi_clean.Berth_share.min(), "-", cppi_clean.Berth_share.max())
print(cppi_clean[["CPPI 2025", "Statistical_index", "Administrative_index"]].corr().round(3).to_string())

ranks: 1 - 400 | duplicated ranks: 0 | ranks match 2025 scores in order: True
rank and score missing together: True
berth share range: 0.369475357710652 - 0.983193806084482
                      CPPI 2025  Statistical_index  Administrative_index
CPPI 2025                 1.000              0.994                 0.999
Statistical_index         0.994              1.000                 0.989
Administrative_index      0.999              0.989                 1.000


In [228]:
# Extreme scores: inspect, keep
ext = cppi_clean.melt(id_vars=["Port", "Country"], value_vars=score_cols, var_name="Year", value_name="CPPI").dropna()
ext["Year"] = ext.Year.str[-4:].astype(int)
print(ext.nsmallest(6, "CPPI").to_string(index=False))
print(ext.nlargest(3, "CPPI").to_string(index=False))


               Port       Country  Year   CPPI
             Durban  South Africa  2024 -721.0
           Savannah United States  2022 -675.0
         Long Beach United States  2021 -665.0
        Los Angeles United States  2021 -598.0
          Cape Town  South Africa  2023 -519.0
Coega (Ngqura) Port  South Africa  2023 -444.0
              Port      Country  Year  CPPI
          Yokohama        Japan  2020 170.0
King Abdullah Port Saudi Arabia  2020 155.0
King Abdullah Port Saudi Arabia  2021 152.0


In [229]:
# Types and long format
for col in ["Rank_2025", "Calls_in_sample", "years_available"]:
    cppi_clean[col] = cppi_clean[col].astype("Int64")
cppi_clean = cppi_clean.reset_index(drop=True)

# one row per port and year
cppi_long = (cppi_clean.melt(id_vars=["Port", "LOCODE", "Country", "ISO3", "WB_region", "Income_group", "Maritime_region"],
                             value_vars=score_cols, var_name="Year", value_name="CPPI").dropna(subset=["CPPI"]))
cppi_long["Year"] = cppi_long.Year.str[-4:].astype(int)
cppi_long = cppi_long.sort_values(["LOCODE", "Year"]).reset_index(drop=True)
print("long format:", cppi_long.shape, "| ports:", cppi_long.LOCODE.nunique(), "| years:", cppi_long.Year.min(), "-", cppi_long.Year.max())

long format: (2212, 9) | ports: 419 | years: 2020 - 2025


In [230]:
# Validate and save
assert cppi_clean.LOCODE.is_unique
assert not cppi_long.duplicated(["LOCODE", "Year"]).any()
cppi_clean.to_csv(PROC / "CPPI_cleaned.csv", index=False)
cppi_long.to_csv(PROC / "CPPI_long.csv", index=False)
print("saved:", cppi_clean.shape, cppi_long.shape)

saved: (419, 22) (2212, 9)


426 rows, of which 7 are blank lines and footnotes.
Only rows with a UN/LOCODE are kept (419 ports).
Columns renamed, text trimmed, curly apostrophes replaced.
LOCODEs are unique and five characters long; each country has one prefix.
ISO3 taken from the LOCODE prefix; all 419 ports get one.
303 ports have all six years; 19 have no 2025 score but do have earlier ones.
2025 ranks run 1–400 in score order; berth share is a fraction, not a percentage.
The lowest scores (Durban, Savannah, Long Beach, Los Angeles) fit known congestion, so they are kept.
Integer columns set; long port-year table added (2,212 rows).
Keys are unique; both tables saved.

#### US.Portcalls Arrivals

In [231]:
# port calls
calls.isnull().sum()

Period                                   0
Period Label                             0
Economy                                  0
Economy Label                            0
CommercialMarket                         0
CommercialMarket Label                   0
Number of port calls                   490
Number of port calls Footnote         9300
Number of port calls Missing value    8810
dtype: int64

In [232]:
# Inspect
print("Original shape:", calls.shape)
print(calls.dtypes.to_string())
print("Footnotes present:", int(calls["Number of port calls Footnote"].notna().sum()))
print("Duplicate economy-period-market records:", calls.duplicated(["Period", "Economy", "CommercialMarket"]).sum())
print("Zero or negative port calls:", int((calls["Number of port calls"] <= 0).sum()))
print(calls["Number of port calls Missing value"].value_counts(dropna=False).to_string())

Original shape: (9300, 9)
Period                                 object
Period Label                           object
Economy                                 int64
Economy Label                          object
CommercialMarket                        int64
CommercialMarket Label                 object
Number of port calls                  float64
Number of port calls Footnote         float64
Number of port calls Missing value     object
Footnotes present: 0
Duplicate economy-period-market records: 0
Zero or negative port calls: 0
Number of port calls Missing value
NaN                                         8810
Not available or not separately reported     310
Not applicable                               108
Not publishable                               72


In [233]:
# Names, period and economy level
calls_clean = calls.drop(columns=["Number of port calls Footnote"]).rename(columns={
    "Period": "Period_code", "Period Label": "Period_label",
    "Economy": "Code", "Economy Label": "Economy",
    "CommercialMarket": "MarketCode", "CommercialMarket Label": "Market",
    "Number of port calls": "Port_calls",
    "Number of port calls Missing value": "Port_calls_missing"})

calls_clean["Year"] = calls_clean["Period_code"].str[:4].astype(int)
calls_clean["Half"] = calls_clean["Period_code"].str[-2:].astype(int)

def level_from_code(code):
    c = int(code)
    return "world" if c == 0 else ("grouping" if c == 1 or c >= 1000 else "country")

calls_clean["level"] = calls_clean["Code"].map(level_from_code)
print(calls_clean.groupby("level").Code.nunique().to_dict())
print(calls_clean.groupby(["Year", "Half"]).Code.nunique().unique())

{'country': 22, 'grouping': 64, 'world': 1}
[87]


In [234]:
# Identifiers are consistent
for a, b in [("Code", "Economy"), ("MarketCode", "Market"), ("Period_code", "Period_label")]:
    print(a, "->", b, "| codes with several names:", int((calls_clean.groupby(a)[b].nunique() > 1).sum()),
          "| names with several codes:", int((calls_clean.groupby(b)[a].nunique() > 1).sum()))

Code -> Economy | codes with several names: 0 | names with several codes: 0
MarketCode -> Market | codes with several names: 0 | names with several codes: 0
Period_code -> Period_label | codes with several names: 0 | names with several codes: 0


In [235]:
# ISO3 for the country rows
# UNCTAD uses its own numeric codes for these four; the ISO numeric codes differ
CODE_FIX = {251: "FRA", 579: "NOR", 926: "GBR", 842: "USA"}

def to_iso3(code):
    if code in CODE_FIX:
        return CODE_FIX[code]
    hit = pycountry.countries.get(numeric=str(code).zfill(3))
    return hit.alpha_3 if hit else None

cn = calls_clean.loc[calls_clean.level == "country", ["Code", "Economy"]].drop_duplicates()
calls_clean["ISO3"] = calls_clean["Code"].map({c: to_iso3(c) for c in cn.Code})
print("countries:", len(cn), "| without ISO3:", cn[cn.Code.map(to_iso3).isna()].Economy.tolist())

countries: 22 | without ISO3: []


In [236]:
# Missing values and their reasons
miss = calls_clean[calls_clean.Port_calls.isna()]
print(miss.groupby(["level", "Port_calls_missing"]).size().to_string())
print("missing without a reason:", int((calls_clean.Port_calls.isna() & calls_clean.Port_calls_missing.isna()).sum()))
print("value present with a reason:", int((calls_clean.Port_calls.notna() & calls_clean.Port_calls_missing.notna()).sum()))
print(miss[miss.level == "country"].groupby("Economy").size().to_string())

level     Port_calls_missing                      
country   Not publishable                              72
grouping  Not applicable                              108
          Not available or not separately reported    310
missing without a reason: 0
value present with a reason: 0
Economy
Singapore    72


In [237]:
# All ships against the sum of the eight ship types
types = calls_clean[calls_clean.MarketCode != 0].groupby(["Period_code", "Code"]).Port_calls.agg(total_types="sum", n_types="count")
allsh = calls_clean[calls_clean.MarketCode == 0].set_index(["Period_code", "Code"]).Port_calls.rename("all_ships")
chk = pd.concat([allsh, types], axis=1).dropna(subset=["all_ships"])
chk = chk[(chk.n_types == 8) & (chk.all_ships > 0)]
chk["gap_pct"] = (chk.total_types / chk.all_ships - 1) * 100
print("checked:", len(chk), "| failures (1% or more):", int((chk.gap_pct.abs() >= 1).sum()))

checked: 830 | failures (1% or more): 0


In [238]:
# Coverage of the country rows against the World total
world = calls_clean[(calls_clean.level == "world") & (calls_clean.MarketCode == 0)].set_index("Period_code").Port_calls
ctry = calls_clean[(calls_clean.level == "country") & (calls_clean.MarketCode == 0)].groupby("Period_code").Port_calls.sum(min_count=1)
cov = (ctry / world).round(3)
print("countries / World, all ships:", cov.min(), "to", cov.max())

countries / World, all ships: 0.791 to 0.82


In [239]:
# Ranges
print("port calls:", calls_clean.Port_calls.min(), "-", calls_clean.Port_calls.max())
print(calls_clean.groupby("Market").Port_calls.median().sort_values(ascending=False).to_string())

port calls: 11.0 - 2461711.0
Market
All ships                           80588.5
Passenger ships                     43772.0
Liquid bulk carriers                12004.5
Container ships                      9848.0
Dry breakbulk carriers               8791.0
Dry bulk carriers                    6612.0
Roll-on/ roll-off ships              3438.0
Liquefied petroleum gas carriers     1190.0
Liquefied natural gas carriers        408.0


In [240]:
# Sort, validate and save
calls_clean = calls_clean.sort_values(["Code", "MarketCode", "Year", "Half"]).reset_index(drop=True)
assert not calls_clean.duplicated(["Period_code", "Code", "MarketCode"]).any()
assert calls_clean.Port_calls.dropna().gt(0).all()

calls_clean.to_csv(PROC / "portcalls_clean.csv", index=False)
print("saved:", PROC / "portcalls_clean.csv", calls_clean.shape)

saved: Datasets\processed\portcalls_clean.csv (9300, 12)


Raw file: 9,300 rows, no footnotes, duplicates or zero values; 490 values are missing.
Names standardised; period split into year and half; economies split into country, grouping and world.
Codes, names and periods map one to one.
ISO3 added for 22 countries; four UNCTAD codes (France, Norway, UK, US) fixed by hand.
Every missing value has a reason; the 72 "not publishable" values are all Singapore.
The eight ship types add up to "All ships" in every complete case.
The 22 countries cover 79–82% of the World total.
Value ranges and median calls per ship type.
Sorted, validated and saved.

#### Daily Chokepoints

In [241]:
# chokepoint data
choke.isnull().sum()

date                      0
year                      0
month                     0
day                       0
portid                    0
portname                  0
n_container               0
n_dry_bulk                0
n_general_cargo           0
n_roro                    0
n_tanker                  0
n_cargo                   0
n_total                   0
capacity_container        0
capacity_dry_bulk         0
capacity_general_cargo    0
capacity_roro             0
capacity_tanker           0
capacity_cargo            0
capacity                  0
ObjectId                  0
dtype: int64

In [242]:
# Inspect
choke = pd.read_csv("Datasets/raw/Daily_Chokepoints_Data.csv")
print("Shape:", choke.shape)
print("Chokepoints:", choke.portid.nunique())
print("Missing values:", int(choke.isnull().sum().sum()))
print("Duplicate chokepoint-day records:", choke.duplicated(["portid", "date"]).sum(), "| ObjectId unique:", choke.ObjectId.is_unique)
print(choke[["portid", "portname"]].drop_duplicates().portname.tolist())

Shape: (80528, 21)
Chokepoints: 28
Missing values: 0
Duplicate chokepoint-day records: 0 | ObjectId unique: True
['Suez Canal', 'Panama Canal', 'Bosporus Strait', 'Bab el-Mandeb Strait', 'Malacca Strait', 'Strait of Hormuz', 'Cape of Good Hope', 'Gibraltar Strait', 'Dover Strait', 'Oresund Strait', 'Taiwan Strait', 'Korea Strait', 'Tsugaru Strait', 'Luzon Strait', 'Lombok Strait', 'Ombai Strait', 'Bohai Strait', 'Torres Strait', 'Sunda Strait', 'Makassar Strait', 'Magellan Strait', 'Yucatan Channel', 'Windward Passage', 'Mona Passage', 'Balabac Strait', 'Bering Strait', 'Mindoro Strait', 'Kerch Strait']


In [243]:
# Dates
ck = choke.copy()
ck["date"] = pd.to_datetime(ck["date"])
parts_match = (ck.date.dt.year.eq(ck.year) & ck.date.dt.month.eq(ck.month) & ck.date.dt.day.eq(ck.day)).mean()
print("year, month, day agree with date:", round(parts_match, 4))
print("period:", ck.date.min().date(), "-", ck.date.max().date())

year, month, day agree with date: 1.0
period: 2019-01-01 - 2026-11-15


In [244]:
# Panel completeness: every chokepoint should have every day
days = ck.date.nunique()
per = ck.groupby("portid").date.nunique()
print("days in period:", days, "| chokepoints with every day:", int((per == days).sum()), "of", len(per))
print("expected rows if complete:", len(per) * days, "| actual:", len(ck))

days in period: 2876 | chokepoints with every day: 28 of 28
expected rows if complete: 80528 | actual: 80528


In [245]:
# Names
ck = ck.rename(columns={"portid": "chokepoint_id", "portname": "chokepoint"})
print("ids with more than one name:", int((ck.groupby("chokepoint_id").chokepoint.nunique() > 1).sum()),
      "| names with more than one id:", int((ck.groupby("chokepoint").chokepoint_id.nunique() > 1).sum()))

ids with more than one name: 0 | names with more than one id: 0


In [246]:
# How the totals are built
types5 = ["container", "dry_bulk", "general_cargo", "roro", "tanker"]
types4 = ["container", "dry_bulk", "general_cargo", "roro"]
n5, n4 = ck[[f"n_{t}" for t in types5]].sum(axis=1), ck[[f"n_{t}" for t in types4]].sum(axis=1)
c5, c4 = ck[[f"capacity_{t}" for t in types5]].sum(axis=1), ck[[f"capacity_{t}" for t in types4]].sum(axis=1)
print("n_total = 5 types:", (ck.n_total == n5).mean(), "| n_cargo = 4 types (no tankers):", (ck.n_cargo == n4).mean())
print("capacity = 5 types: largest gap", int((ck.capacity - c5).abs().max()), "| capacity_cargo = 4 types: largest gap", int((ck.capacity_cargo - c4).abs().max()), "(rounding)")

n_total = 5 types: 1.0 | n_cargo = 4 types (no tankers): 1.0
capacity = 5 types: largest gap 4 | capacity_cargo = 4 types: largest gap 3 (rounding)


In [247]:
# Zero days and capacity gaps
ck["no_transits"] = ck.n_total.eq(0)
ck["capacity_missing"] = ck.n_total.gt(0) & ck.capacity.eq(0)
print("days with no transits:", int(ck.no_transits.sum()))
print(ck[ck.no_transits].groupby("chokepoint").size().sort_values(ascending=False).head(4).to_string())
print("days with transits but zero capacity:", int(ck.capacity_missing.sum()))
print(ck[ck.capacity_missing].groupby("chokepoint").size().sort_values(ascending=False).head(4).to_string())

days with no transits: 2168
chokepoint
Bering Strait       1781
Kerch Strait         296
Magellan Strait       72
Strait of Hormuz      14
days with transits but zero capacity: 449
chokepoint
Bering Strait       263
Magellan Strait      96
Kerch Strait         51
Strait of Hormuz     17


In [248]:
# Values and shifts to be aware of
num = ck.select_dtypes("number").drop(columns=["year", "month", "day", "ObjectId"])
print("negative values:", int((num < 0).sum().sum()))
print(ck.groupby("chokepoint").n_total.agg(["median", "max"]).sort_values("median", ascending=False).head(5).to_string())

ck["year_"] = ck.date.dt.year
print(ck[ck.chokepoint.isin(["Suez Canal", "Bab el-Mandeb Strait", "Cape of Good Hope", "Panama Canal"])]
      .groupby(["chokepoint", "year_"]).n_total.mean().unstack().round(1).to_string())

hz = ck[ck.chokepoint == "Strait of Hormuz"].groupby(ck.date.dt.to_period("M")).n_total.mean().round(0)
print("Strait of Hormuz, average transits per day:", hz.tail(10).to_dict())

negative values: 0
                median  max
chokepoint                 
Taiwan Strait    245.0  472
Korea Strait     230.0  448
Malacca Strait   206.0  298
Bohai Strait     179.0  459
Dover Strait     167.0  244
year_                 2019  2020  2021  2022  2023  2024  2025  2026
chokepoint                                                          
Bab el-Mandeb Strait  53.5  54.5  58.8  67.1  74.0  32.5  33.5  32.7
Cape of Good Hope     48.1  52.8  46.9  43.8  49.1  87.0  89.5  89.8
Panama Canal          32.0  31.8  32.5  33.4  30.2  26.5  31.1  31.1
Suez Canal            52.5  52.8  57.7  67.1  73.7  39.6  38.5  41.1
Strait of Hormuz, average transits per day: {Period('2026-02', 'M'): 78.0, Period('2026-03', 'M'): 3.0, Period('2026-04', 'M'): 7.0, Period('2026-05', 'M'): 4.0, Period('2026-06', 'M'): 13.0, Period('2026-07', 'M'): 10.0, Period('2026-08', 'M'): 5.0, Period('2026-09', 'M'): 4.0, Period('2026-10', 'M'): 4.0, Period('2026-11', 'M'): 3.0}


In [249]:
# Partial last year
ck["partial_year"] = ck.date.dt.year.eq(ck.date.max().year)
print("partial year:", ck.date.max().year, "| last date:", ck.date.max().date())

partial year: 2026 | last date: 2026-11-15


In [250]:
# Tidy and save
ck_clean = ck.drop(columns=["year", "month", "day", "ObjectId", "year_"]).sort_values(["chokepoint_id", "date"]).reset_index(drop=True)
assert not ck_clean.duplicated(["chokepoint_id", "date"]).any()
print("Cleaned shape:", ck_clean.shape)

ck_clean.to_csv(PROC / "chokepoints_cleaned.csv", index=False)
ck_clean.drop_duplicates("chokepoint_id")[["chokepoint_id", "chokepoint"]].to_csv(PROC / "chokepoints_list.csv", index=False)
print("saved:", ck_clean.shape)

Cleaned shape: (80528, 20)
saved: (80528, 20)


Raw file: 80,528 rows for 28 chokepoints, no missing values or duplicates.
`date` converted; year, month and day agree with it. The period runs from 1 January 2019 to 15 November 2026.
Every chokepoint has every one of the 2,876 days, so the panel is complete.
Columns renamed to `chokepoint_id` and `chokepoint` so they are not confused with ports; each id has one name.
`n_total` is the sum of five vessel types and `n_cargo` the sum of four (no tankers), exactly. Capacity totals match their parts to within rounding (at most 4 units).
2,168 days have no transits, mostly Bering Strait (ice season) and Kerch Strait; kept as real zeros. 449 days have transits but zero capacity, flagged `capacity_missing`.
No negative values. Suez and Bab el-Mandeb fall by about half in 2024 while the Cape of Good Hope rises by about 80%. The Strait of Hormuz drops from about 78 transits a day in February 2026 to under 15 from March; this is kept as reported and will be examined in the disruption analysis.
2026 is flagged as a partial year (data ends 15 November).
Redundant columns dropped, data sorted, duplicates checked, table and chokepoint list saved.

## EDA

In [251]:
# Seaborne


# Modelling 

# Evaluation

# Deployment